In [ ]:
# ─── CELL 1: 패키지 설치 ─────────────────────────────────
!pip install -q transformers torch torchvision stable-baselines3 shimmy \
             scikit-learn pandas numpy matplotlib joblib pyarrow scipy ftfy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 39.1 MB/s eta 0:00:00


In [ ]:
!pip install -q "torchao>=0.16.0" --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 99.5 MB/s eta 0:00:00


In [ ]:
# ─── CELL 2: Drive 마운트 + Rev4 폴더 구조 생성/정리 ──────
import os, shutil, sys
from google.colab import drive

drive.mount('/content/drive')

REV3_ROOT = '/content/drive/MyDrive/X-MultiVLA_rev2_5Simbols'
REV4_ROOT = '/content/drive/MyDrive/X-MultiVLA_rev4'
REV4_DATA = f'{REV4_ROOT}/data'
REV4_SRC  = f'{REV4_ROOT}/src'

# ── 폴더 구조 생성 ───────────────────────────────────────
for d in [REV4_DATA,
          f'{REV4_SRC}/models', f'{REV4_SRC}/training',
          f'{REV4_SRC}/data',   f'{REV4_SRC}/utils',
          f'{REV4_ROOT}/checkpoints', f'{REV4_ROOT}/outputs', f'{REV4_ROOT}/logs']:
    os.makedirs(d, exist_ok=True)

for d in [f'{REV4_SRC}/models', f'{REV4_SRC}/training',
          f'{REV4_SRC}/data',   f'{REV4_SRC}/utils']:
    open(f'{d}/__init__.py', 'w').close()

# ── Rev3 데이터 처리 코드 → src/data/ 복사 ──────────────
copies = [
    ('data/collector.py',                f'{REV4_SRC}/data/collector.py'),
    ('data/feature_engineer.py',         f'{REV4_SRC}/data/feature_engineer.py'),
    ('data/multi_asset_preprocessor.py', f'{REV4_SRC}/data/preprocessor.py'),
    ('utils/portfolio_evaluator.py',     f'{REV4_SRC}/utils/evaluator.py'),
]
for src_rel, dst in copies:
    src = f'{REV3_ROOT}/{src_rel}'
    if os.path.exists(src):
        shutil.copy(src, dst); print(f'  ✅ 복사: {os.path.basename(src_rel)} → src/')

# ── 뉴스 데이터 → data/ 복사 ────────────────────────────
for fname in ['news_all_8h.csv', 'news_sentiment_8h.csv']:
    src = f'{REV3_ROOT}/data/{fname}'
    dst = f'{REV4_DATA}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst); print(f'  ✅ 복사: {fname} → data/')
    elif os.path.exists(dst):
        print(f'  ✅ 이미 존재: {fname}')

# ── 잘못된 위치 파일 정리 (src/ 밖 구버전) ──────────────
stale = [
    f'{REV4_ROOT}/models',
    f'{REV4_ROOT}/utils',
    f'{REV4_ROOT}/data/collector.py',
    f'{REV4_ROOT}/data/feature_engineer.py',
    f'{REV4_ROOT}/data/preprocessor.py',
]
for path in stale:
    if os.path.isdir(path):
        shutil.rmtree(path); print(f'  🗑️  삭제: {path.replace(REV4_ROOT,"")}/')
    elif os.path.isfile(path):
        os.remove(path); print(f'  🗑️  삭제: {path.replace(REV4_ROOT,"")}')

sys.path.insert(0, REV4_SRC)
sys.path.insert(0, REV4_ROOT)

print(f'\n✅ Rev4 폴더 정리 완료')
print(f'   구조: data/ (파일만)  src/ (코드만)')

Mounted at /content/drive
  ✅ 복사: collector.py → src/
  ✅ 복사: feature_engineer.py → src/
  ✅ 복사: multi_asset_preprocessor.py → src/
  ✅ 복사: portfolio_evaluator.py → src/
  ✅ 이미 존재: news_all_8h.csv
  ✅ 이미 존재: news_sentiment_8h.csv

✅ Rev4 폴더 정리 완료
   구조: data/ (파일만)  src/ (코드만)


In [ ]:
# ─── CELL 3: src/models/finbert_encoder.py 작성 ──────────
code = '''import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

class FinBERTEncoder(nn.Module):
    """
    FinBERT 768d 히든스테이트 인코더

    백본 동결(Backbone Freeze):
    - FinBERT는 이미 금융 텍스트로 충분히 학습된 전문가
    - requires_grad=False로 가중치 완전 동결
    - 뉴스 텍스트 → 768d 임베딩 변환기로만 사용
    - VRAM 절약 + Catastrophic Forgetting 방지
    - 학습 대상: Cross-Attention, LLM LoRA, Action Head만
    """
    def __init__(self, device: str = "cpu", max_length: int = 128):
        super().__init__()
        self.tokenizer  = AutoTokenizer.from_pretrained("ProsusAI/finbert")
        self.model      = AutoModel.from_pretrained("ProsusAI/finbert")
        self.device     = device
        self.max_length = max_length

        # ★ 백본 동결: 모든 파라미터 학습 차단
        for param in self.model.parameters():
            param.requires_grad = False

        self.model.to(device)
        self.model.eval()  # 항상 eval 모드 유지

    def encode_texts(self, texts: list) -> torch.Tensor:
        """texts → (B, 768) CLS 히든스테이트"""
        enc = self.tokenizer(
            texts, padding=True, truncation=True,
            max_length=self.max_length, return_tensors="pt"
        ).to(self.device)
        with torch.no_grad():  # 그래디언트 계산 완전 차단
            out = self.model(**enc)
        return out.last_hidden_state[:, 0, :]  # (B, 768)

    def encode_bucket(self, texts: list) -> torch.Tensor:
        """8h 버킷 내 뉴스들 → 평균 풀링 → (768,)"""
        if not texts:
            return torch.zeros(768, device=self.device)
        return self.encode_texts(texts).mean(dim=0)

    def encode_timeseries(self, news_by_bucket: list) -> torch.Tensor:
        """타임스텝별 뉴스 버킷 리스트 → (T, 768)"""
        return torch.stack([self.encode_bucket(b) for b in news_by_bucket])

    def train(self, mode=True):
        """동결 상태 유지 — train() 호출해도 eval 모드 고수"""
        super().train(mode)
        self.model.eval()  # FinBERT는 항상 eval
        return self
'''
with open(f'{REV4_SRC}/models/finbert_encoder.py', 'w') as f:
    f.write(code)
print('✅ src/models/finbert_encoder.py')
print('   FinBERT 백본 완전 동결 (requires_grad=False)')
print('   train() 호출해도 eval 모드 유지')

✅ src/models/finbert_encoder.py
   FinBERT 백본 완전 동결 (requires_grad=False)
   train() 호출해도 eval 모드 유지


In [ ]:
# ─── CELL 4: src/models/cross_attention.py 작성 ──────────
code = '''import torch
import torch.nn as nn

class CrossModalAttention(nn.Module):
    """
    양방향 Cross-Attention (전문가 리뷰 반영)

    단방향 문제:
    - V→L 만 있으면: 가격이 평온할 때 블랙스완 뉴스 어텐션 낮아짐
    해결:
    - V→L: 가격이 뉴스에 질문 (기존)
    - L→V: 뉴스가 가격에 질문 (신규) → 강한 뉴스가 가격 해석 강제
    - 두 방향 결합 → 대칭 융합
    """
    def __init__(self, v_dim=256, l_dim=768, d_model=256, n_heads=8, dropout=0.1):
        super().__init__()

        # V→L 방향 (가격이 뉴스에 질문)
        self.v_proj_q  = nn.Linear(v_dim, d_model)
        self.l_proj_kv = nn.Linear(l_dim, d_model)
        self.attn_v2l  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)

        # L→V 방향 (뉴스가 가격에 질문) — 블랙스완 대응
        self.l_proj_q  = nn.Linear(l_dim, d_model)
        self.v_proj_kv = nn.Linear(v_dim, d_model)
        self.attn_l2v  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)

        # 두 방향 결합
        self.combine   = nn.Linear(d_model * 2, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff    = nn.Sequential(nn.Linear(d_model, d_model*2), nn.GELU(),
                                    nn.Dropout(dropout), nn.Linear(d_model*2, d_model))
        self.drop  = nn.Dropout(dropout)

    def forward(self, v_emb: torch.Tensor, l_emb: torch.Tensor):
        """
        v_emb: (B, v_dim)
        l_emb: (B, T_l, l_dim) 또는 (B, l_dim)
        반환:
          fused:    (B, d_model)   — 양방향 융합 표현
          attn_v2l: (B, 1, T_l)   — V→L 어텐션 가중치 (XAI용)
          attn_l2v: (B, 1, 1)     — L→V 어텐션 가중치 (XAI용)
        """
        if l_emb.dim() == 2:
            l_emb = l_emb.unsqueeze(1)          # (B, 1, l_dim)

        v_q  = self.v_proj_q(v_emb).unsqueeze(1)    # (B, 1, d)
        l_kv = self.l_proj_kv(l_emb)                # (B, T_l, d)

        # ── V→L: 가격이 뉴스에 질문 ───────────────────────
        out_v2l, attn_v2l = self.attn_v2l(v_q, l_kv, l_kv)
        out_v2l = out_v2l.squeeze(1)                 # (B, d)

        # ── L→V: 뉴스가 가격에 질문 ───────────────────────
        l_q_mean = l_emb.mean(dim=1, keepdim=True)   # (B, 1, l_dim) 뉴스 평균
        l_q      = self.l_proj_q(l_q_mean)            # (B, 1, d)
        v_kv     = self.v_proj_kv(v_emb).unsqueeze(1) # (B, 1, d)
        out_l2v, attn_l2v = self.attn_l2v(l_q, v_kv, v_kv)
        out_l2v = out_l2v.squeeze(1)                  # (B, d)

        # ── 양방향 결합 ────────────────────────────────────
        fused = self.combine(torch.cat([out_v2l, out_l2v], dim=-1))  # (B, d)
        fused = self.norm1(fused + v_q.squeeze(1))
        fused = self.norm2(fused + self.drop(self.ff(fused)))

        return fused, attn_v2l, attn_l2v
'''
with open(f'{REV4_SRC}/models/cross_attention.py', 'w') as f:
    f.write(code)
print('✅ src/models/cross_attention.py (양방향 Cross-Attention)')

✅ src/models/cross_attention.py (양방향 Cross-Attention)


In [ ]:
# ─── CELL 5: src/models/action_head.py 작성 ─────────────
code = '''import torch
import torch.nn as nn
import torch.nn.functional as F

class ActionHead(nn.Module):
    """
    L 전략 임베딩 → 포트폴리오 비중 (인터페이스 어댑터)
    전략은 L이 결정, Head는 인터페이스별 포맷 변환만
    """
    def __init__(self, in_dim: int, n_assets: int = 5, dropout: float = 0.1):
        super().__init__()
        self.n_assets = n_assets
        self.decoder  = nn.Sequential(
            nn.Linear(in_dim, 128), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),    nn.GELU(),
            nn.Linear(64, n_assets + 1),  # +1 현금
        )

    def forward(self, x: torch.Tensor, nav: dict = None) -> torch.Tensor:
        weights = F.softmax(self.decoder(x), dim=-1)
        if nav:
            weights = self._apply_nav(weights, nav)
        return weights

    def _apply_nav(self, w, nav):
        w = w.clone()
        min_cash = nav.get("min_cash", 0.0)
        if min_cash > 0:
            deficit = (min_cash - w[:, -1]).clamp(min=0)
            w[:, -1] += deficit
            s = w[:, :self.n_assets].sum(-1, keepdim=True).clamp(min=1e-8)
            w[:, :self.n_assets] -= deficit.unsqueeze(-1) * w[:, :self.n_assets] / s
        max_s = nav.get("max_single", 1.0)
        if max_s < 1.0:
            w = w.clamp(max=max_s)
            w = w / w.sum(-1, keepdim=True).clamp(min=1e-8)
        return w
'''
with open(f'{REV4_SRC}/models/action_head.py', 'w') as f:
    f.write(code)
print('✅ src/models/action_head.py')

✅ src/models/action_head.py


In [ ]:
# ─── CELL 6: src/models/llm_reasoning.py 작성 ────────────
code = '''import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, AutoConfig
from peft import get_peft_model, LoraConfig

DEFAULT_LLM   = "Qwen/Qwen2.5-1.5B"
SYSTEM_PROMPT = (
    "You are a crypto portfolio manager. "
    "Analyze the market state and news to decide portfolio allocation."
)

class VProjector(nn.Module):
    def __init__(self, v_dim, hidden_size):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(v_dim, hidden_size), nn.GELU(),
            nn.Linear(hidden_size, hidden_size), nn.LayerNorm(hidden_size))
    def forward(self, x): return self.proj(x)

class LLMReasoningModule(nn.Module):
    def __init__(self, v_dim=256, llm_name=DEFAULT_LLM,
                 device="cpu", use_lora=True, lora_r=16, lora_alpha=32):
        super().__init__()
        self.device = device
        cfg = AutoConfig.from_pretrained(llm_name)
        self.hidden_size = cfg.hidden_size

        self.tokenizer = AutoTokenizer.from_pretrained(llm_name, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # fp32로 로드 (dtype 충돌 방지 — A100 40GB에서 1.5B fp32 = ~6GB)
        base = AutoModel.from_pretrained(
            llm_name, trust_remote_code=True,
            torch_dtype=torch.float32)

        if use_lora:
            base = get_peft_model(base, LoraConfig(
                r=lora_r, lora_alpha=lora_alpha,
                target_modules=["q_proj","k_proj","v_proj","o_proj"],
                lora_dropout=0.05, bias="none"))
        self.llm = base.to(device)
        self.v_projector = VProjector(v_dim, self.hidden_size).to(device)

    def forward(self, v_emb, news_texts):
        B = v_emb.size(0)
        # 모두 float32 — dtype 충돌 없음
        v_token = self.v_projector(v_emb.float().to(self.device)).unsqueeze(1)

        prompts = [f"{SYSTEM_PROMPT}\\nNews: {t}\\nAllocation:" for t in news_texts]
        enc = self.tokenizer(prompts, return_tensors="pt", padding=True,
                             truncation=True, max_length=256).to(self.device)
        text_embs = self.llm.get_input_embeddings()(enc.input_ids)  # float32
        combined  = torch.cat([v_token, text_embs], dim=1)
        mask      = torch.cat([torch.ones(B,1,device=self.device,dtype=torch.long),
                                enc.attention_mask], dim=1)
        out = self.llm(inputs_embeds=combined, attention_mask=mask)
        return out.last_hidden_state[:, 0, :]  # (B, H) float32
'''
with open(f'{REV4_SRC}/models/llm_reasoning.py', 'w') as f:
    f.write(code)
print('✅ src/models/llm_reasoning.py — fp32으로 통일, dtype 충돌 해결')

✅ src/models/llm_reasoning.py — fp32으로 통일, dtype 충돌 해결


In [ ]:
# ─── CELL 6: src/models/itransformer.py 작성 ─────────────
code = '''import torch
import torch.nn as nn
import torch.nn.functional as F

class iTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(nn.Linear(d_model, d_model*4), nn.GELU(),
                                    nn.Dropout(dropout), nn.Linear(d_model*4, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        a, _ = self.attn(x, x, x)
        x = self.norm1(x + self.drop(a))
        x = self.norm2(x + self.drop(self.ff(x)))
        return x


class iTransformer(nn.Module):
    """
    iTransformer — V 컴포넌트 (ICLR 2024)
    코인 간 어텐션 + 헤딩값(예측 수익률) + 불확실성 출력

    개선사항 (전문가 리뷰 반영):
    - Point Estimate 대신 (mu, sigma) 쌍 출력
    - LLM이 "불확실성 높음 → 포지션 축소" 판단 가능
    """
    def __init__(self, n_coins=5, n_features=94, seq_len=60,
                 d_model=256, n_heads=8, n_layers=4, dropout=0.1):
        super().__init__()
        self.n_coins = n_coins
        self.d_model = d_model

        self.token_embed = nn.Linear(seq_len * n_features, d_model)
        self.pos_embed   = nn.Parameter(torch.randn(1, n_coins, d_model) * 0.02)
        self.drop        = nn.Dropout(dropout)
        self.layers      = nn.ModuleList([
            iTransformerLayer(d_model, n_heads, dropout) for _ in range(n_layers)
        ])
        self.norm        = nn.LayerNorm(d_model)

        # 헤딩값: 예측 수익률 (방향 + 크기)
        self.heading_mu  = nn.Linear(d_model, 1)

        # 불확실성: 예측 신뢰구간 (Softplus → 항상 양수)
        self.heading_sig = nn.Sequential(nn.Linear(d_model, 1), nn.Softplus())

        # V 임베딩: Cross-Attention 입력용
        self.v_proj      = nn.Linear(d_model * n_coins, d_model)

    def forward(self, x: torch.Tensor):
        """
        x: (B, T, N, F)
        반환:
          heading_mu:  (B, N)      — 예측 수익률
          heading_sig: (B, N)      — 불확실성 (클수록 신뢰도 낮음)
          v_emb:       (B, d_model) — V 임베딩
        """
        B, T, N, F = x.shape
        tokens = x.permute(0, 2, 1, 3).reshape(B, N, T * F)
        tokens = self.drop(self.token_embed(tokens) + self.pos_embed)

        for layer in self.layers:
            tokens = layer(tokens)
        tokens = self.norm(tokens)  # (B, N, d)

        heading_mu  = self.heading_mu(tokens).squeeze(-1)   # (B, N)
        heading_sig = self.heading_sig(tokens).squeeze(-1)  # (B, N) 양수

        v_emb = self.v_proj(tokens.reshape(B, N * self.d_model))  # (B, d)
        return heading_mu, heading_sig, v_emb
'''
with open(f'{REV4_SRC}/models/itransformer.py', 'w') as f:
    f.write(code)
print('✅ src/models/itransformer.py (불확실성 출력 추가)')

✅ src/models/itransformer.py (불확실성 출력 추가)


In [ ]:
# ─── CELL 8: src/models/vla_model.py 작성 ────────────────
code = '''import torch
import torch.nn as nn
from models.itransformer import iTransformer
from models.cross_attention import CrossModalAttention
from models.llm_reasoning import LLMReasoningModule
from models.action_head import ActionHead

class VLAModel(nn.Module):
    """
    X-MultiVLA Rev4 — E2E VLA 모델

    Phase 1 (Oracle SL):  iTransformer + CrossAttn + phase1_proj + ActionHead
                          (LLM 건너뜀 → NaN 방지, 빠름)
    Phase 2 (GRPO):       iTransformer 동결 + CrossAttn + LLM + ActionHead
    """
    def __init__(self, n_coins=5, n_features=106, seq_len=60,
                 d_model=256, n_heads=8, n_layers_v=4,
                 l_dim=768, n_heads_ca=8,
                 llm_name="Qwen/Qwen2.5-1.5B",
                 lora_r=16, lora_alpha=32,
                 n_assets=5, dropout=0.1,
                 device="cpu", use_lora=True):
        super().__init__()
        self.device   = device
        self.n_assets = n_assets

        self.v_encoder  = iTransformer(
            n_coins=n_coins, n_features=n_features, seq_len=seq_len,
            d_model=d_model, n_heads=n_heads, n_layers=n_layers_v, dropout=dropout)

        self.cross_attn = CrossModalAttention(
            v_dim=d_model, l_dim=l_dim, d_model=d_model,
            n_heads=n_heads_ca, dropout=dropout)

        # Phase 1 전용: LLM 대신 단순 프로젝션 (안정적, 빠름)
        self.phase1_proj = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model * 2, d_model))
        self.phase1_head = ActionHead(d_model, n_assets, dropout)

        # Phase 2 전용: LLM + ActionHead
        self.llm = LLMReasoningModule(
            v_dim=d_model, llm_name=llm_name, device=device,
            use_lora=use_lora, lora_r=lora_r, lora_alpha=lora_alpha)
        self.action_head = ActionHead(self.llm.hidden_size, n_assets, dropout)

        self.v_norm = nn.LayerNorm(d_model)
        self.l_norm = nn.LayerNorm(l_dim)

    # ── Phase 1: LLM 없는 순방향 (안정적) ─────────────────
    def forward_phase1(self, x, l_emb):
        """
        Phase 1 Oracle SL 전용
        x:     (B, T, N, F)
        l_emb: (B, 5, 768)
        """
        _, _, v_emb = self.v_encoder(x.to(self.device))
        v_emb  = self.v_norm(v_emb)
        l_emb  = self.l_norm(l_emb.to(self.device))
        fused, _, _ = self.cross_attn(v_emb, l_emb)
        out    = self.phase1_proj(fused)
        return self.phase1_head(out)

    # ── Phase 2: LLM 포함 순방향 ──────────────────────────
    def forward_from_v_emb(self, v_emb, l_emb, news_texts, nav=None):
        """
        Phase 2 GRPO: 사전계산된 v_emb 직접 입력
        v_emb: (B, d_model)  l_emb: (B, 5, 768)
        """
        v_emb  = self.v_norm(v_emb.to(self.device))
        l_emb  = self.l_norm(l_emb.to(self.device))
        fused, attn_v2l, _ = self.cross_attn(v_emb, l_emb)
        hidden  = self.llm(fused, news_texts)
        weights = self.action_head(hidden, nav)
        return weights, hidden, attn_v2l

    def freeze_v_encoder(self):
        for p in self.v_encoder.parameters(): p.requires_grad_(False)
        print("✅ iTransformer 동결")

    def trainable_parameters(self):
        return [p for p in self.parameters() if p.requires_grad]

    def print_params(self):
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.trainable_parameters())
        print(f"  iTransformer:  {sum(p.numel() for p in self.v_encoder.parameters()):,}")
        print(f"  CrossAttn:     {sum(p.numel() for p in self.cross_attn.parameters()):,}")
        print(f"  Phase1 Proj:   {sum(p.numel() for p in self.phase1_proj.parameters()):,}")
        print(f"  LLM LoRA:      {sum(p.numel() for p in self.llm.parameters() if p.requires_grad):,}")
        print(f"  전체: {total:,}  학습: {trainable:,} ({100*trainable/total:.1f}%)")
'''
with open(f'{REV4_SRC}/models/vla_model.py', 'w') as f:
    f.write(code)
open(f'{REV4_SRC}/models/__init__.py', 'w').close()
print('✅ vla_model.py — Phase1 LLM 분리 (NaN 근본 해결)')

✅ vla_model.py — Phase1 LLM 분리 (NaN 근본 해결)


In [ ]:
# [XAI 버전 - 선택실행] ────────────────────────────────────
# ─── CELL 8-XAI: XAI 모델 구조 검증 (CPU) ────────────────
import torch, sys
sys.path.insert(0, REV4_ROOT)
from models.cross_attention_fusion import CrossModalFusion, XAIReasoningModule
from models.action_head import ActionHead
from models.vla_model import VLAAgent

B = 4
v_emb = torch.randn(B, 640)
l_emb = torch.randn(B, 5, 768)
model = VLAAgent(embed_dim=640, l_dim=768, d_model=256, n_heads=8, n_assets=5)
weights, regime_logits, attn_weights = model(v_emb, l_emb)
print('✅ [XAI] VLA 모델 검증 완료')
print(f'   weights: {weights.shape}, regime: {regime_logits.shape}, attn: {attn_weights.shape}')
n = sum(p.numel() for p in model.parameters())
print(f'   파라미터: {n:,} ({n/1e6:.1f}M)')

In [ ]:
# ─── CELL 9: 환경 검증 (CPU) ─────────────────────────────
import numpy as np
from utils.portfolio_env import VLAPortfolioEnv

T = 100
v_dim, l_dim, n_assets = 640, 768, 5

v_embs  = np.random.randn(T, v_dim).astype(np.float32)
l_embs  = np.random.randn(T, l_dim).astype(np.float32)
prices  = np.abs(np.random.randn(T+1, n_assets)).astype(np.float64) + 1.0

# 기본값 (최대 수익률, 제약 없음)
env = VLAPortfolioEnv(v_embs, l_embs, prices, nav_constraints={})
obs, _ = env.reset()
print(f'✅ 환경 검증 완료')
print(f'   관측 차원: {obs.shape}  (V {v_dim}d + L {l_dim}d = {v_dim+l_dim}d)')

total_reward = 0
for _ in range(10):
    action = env.action_space.sample()
    obs, reward, done, _, info = env.step(action)
    total_reward += reward
print(f'   10스텝 누적보상: {total_reward:.4f}')
print(f'   최종 자산가치: {info["equity"]:.4f}')

# Navigation 컨스트레인트 테스트
env_nav = VLAPortfolioEnv(v_embs, l_embs, prices,
                           nav_constraints={"min_cash": 0.2, "max_single": 0.4})
obs, _ = env_nav.reset()
obs, reward, _, _, info = env_nav.step(np.ones(n_assets+1))
print(f'\n   Nav 테스트 (현금≥20%, 단일자산≤40%)')
print(f'   현금 비중: {info["weights"][-1]:.3f} (≥0.2)')
print(f'   최대 비중: {info["weights"].max():.3f} (≤0.4)')

In [ ]:
# ═══════════════════════════════════════════════════════
# [LLM 버전] CELL 10~13 — Qwen2.5-1.5B 기반 L 컴포넌트
# ═══════════════════════════════════════════════════════
# ─── CELL 10: LLM 패키지 추가 설치 ───────────────────────
!pip install -q peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.7 MB/s eta 0:00:00


In [ ]:
# ─── CELL 11: models/llm_reasoning.py 작성 ───────────────
code = '''import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, AutoConfig
from peft import get_peft_model, LoraConfig, TaskType

# 기본 LLM 모델명 (변경 가능)
DEFAULT_LLM = "Qwen/Qwen2.5-1.5B"

SYSTEM_PROMPT = (
    "You are a crypto portfolio manager. "
    "Analyze the market state and news to decide portfolio allocation."
)

class VProjector(nn.Module):
    """PatchTST 임베딩 → LLM 토큰 공간으로 투영"""
    def __init__(self, v_dim: int, hidden_size: int):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(v_dim, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
        )

    def forward(self, v_emb: torch.Tensor) -> torch.Tensor:
        return self.proj(v_emb)  # (B, hidden_size)


class LLMReasoningModule(nn.Module):
    """
    LLM 기반 L 컴포넌트
    - V 임베딩을 LLM 토큰 공간에 투영 → 뉴스 텍스트 토큰과 결합
    - LLM이 [V토큰 | 시스템프롬프트 | 뉴스] 를 함께 처리
    - 마지막 히든스테이트 → Action Head로 전달
    - LoRA로 효율적 파인튜닝
    """
    def __init__(self, v_dim: int = 640, llm_name: str = DEFAULT_LLM,
                 device: str = "cpu", use_lora: bool = True,
                 lora_r: int = 16, lora_alpha: int = 32):
        super().__init__()
        self.device = device

        # ── LLM 로드 ────────────────────────────────────────
        config = AutoConfig.from_pretrained(llm_name)
        self.hidden_size = config.hidden_size

        self.tokenizer = AutoTokenizer.from_pretrained(
            llm_name, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        llm_base = AutoModel.from_pretrained(
            llm_name, trust_remote_code=True,
            torch_dtype=torch.float16 if device != "cpu" else torch.float32,
        )

        # ── LoRA 적용 ────────────────────────────────────────
        if use_lora:
            lora_cfg = LoraConfig(
                r=lora_r,
                lora_alpha=lora_alpha,
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
                lora_dropout=0.05,
                bias="none",
            )
            self.llm = get_peft_model(llm_base, lora_cfg)
        else:
            self.llm = llm_base

        self.llm.to(device)

        # ── V 투영기 (학습 가능) ─────────────────────────────
        self.v_projector = VProjector(v_dim, self.hidden_size).to(device)

    def forward(self, v_emb: torch.Tensor,
                news_texts: list[str]) -> torch.Tensor:
        """
        v_emb:      (B, v_dim)   — PatchTST 출력
        news_texts: List[str]    — 배치 크기 B의 뉴스 문자열

        반환: (B, hidden_size) — LLM 마지막 히든스테이트 (V토큰 위치)
        """
        B = v_emb.size(0)

        # 1. V 임베딩 → LLM 토큰 공간
        v_token = self.v_projector(v_emb.to(self.device))  # (B, hidden_size)

        # 2. 텍스트 프롬프트 토크나이징
        prompts = [f"{SYSTEM_PROMPT}\\nNews: {txt}\\nAllocation:" for txt in news_texts]
        enc = self.tokenizer(
            prompts, return_tensors="pt",
            padding=True, truncation=True, max_length=256,
        ).to(self.device)

        # 3. 텍스트 토큰 임베딩
        text_embs = self.llm.get_input_embeddings()(enc.input_ids)  # (B, T, H)

        # 4. [V토큰 | 텍스트토큰] 결합
        v_token_seq = v_token.unsqueeze(1)                           # (B, 1, H)
        combined    = torch.cat([v_token_seq, text_embs], dim=1)     # (B, 1+T, H)

        # 5. 어텐션 마스크 확장
        v_mask         = torch.ones(B, 1, device=self.device, dtype=torch.long)
        attention_mask = torch.cat([v_mask, enc.attention_mask], dim=1)  # (B, 1+T)

        # 6. LLM 포워드
        outputs = self.llm(
            inputs_embeds=combined,
            attention_mask=attention_mask,
            output_hidden_states=False,
        )
        # V토큰(index 0) 위치의 마지막 히든스테이트 반환
        last_hidden = outputs.last_hidden_state[:, 0, :]  # (B, H)

        return last_hidden.float()
'''

with open(f'{REV4_ROOT}/models/llm_reasoning.py', 'w') as f:
    f.write(code)
print('✅ [LLM] models/llm_reasoning.py 작성 완료')
print(f'   기본 모델: Qwen/Qwen2.5-1.5B (LoRA r=16)')
print(f'   V 투영: PatchTST 640d → LLM hidden_size')

✅ [LLM] models/llm_reasoning.py 작성 완료
   기본 모델: Qwen/Qwen2.5-1.5B (LoRA r=16)
   V 투영: PatchTST 640d → LLM hidden_size


In [ ]:
# ─── CELL 12: models/vla_model_llm.py 작성 ───────────────
code = '''import torch
import torch.nn as nn
from models.llm_reasoning import LLMReasoningModule
from models.action_head import ActionHead

class VLAModelLLM(nn.Module):
    """
    LLM 기반 E2E VLA 모델
    V (PatchTST) + L (Qwen LLM + 뉴스) → Action Head → 포트폴리오 비중

    구조:
      [V 임베딩] → VProjector ──┐
                                ├→ LLM Backbone → [V토큰 히든] → Action Head → 비중
      [뉴스 텍스트] → 토크나이저 ─┘
    """
    def __init__(self, v_dim: int = 640, n_assets: int = 5,
                 llm_name: str = "Qwen/Qwen2.5-1.5B",
                 device: str = "cpu", use_lora: bool = True,
                 lora_r: int = 16, dropout: float = 0.1):
        super().__init__()

        self.llm_module = LLMReasoningModule(
            v_dim=v_dim, llm_name=llm_name,
            device=device, use_lora=use_lora, lora_r=lora_r,
        )
        hidden_size = self.llm_module.hidden_size

        # Action Head: LLM 히든스테이트 → 포트폴리오 비중
        self.action_head = ActionHead(hidden_size, n_assets, dropout)
        self.n_assets = n_assets

    def forward(self, v_emb: torch.Tensor, news_texts: list[str],
                nav_constraints: dict = None):
        """
        v_emb:      (B, v_dim)
        news_texts: List[str], 길이 B
        반환:
          weights:     (B, n_assets+1) — 포트폴리오 비중 (합=1)
          last_hidden: (B, hidden_size) — UX LLM 설명용
        """
        last_hidden = self.llm_module(v_emb, news_texts)      # (B, H)
        weights     = self.action_head(last_hidden, nav_constraints)
        return weights, last_hidden

    def trainable_parameters(self):
        """학습 가능한 파라미터만 반환 (LoRA + VProjector + ActionHead)"""
        return [p for p in self.parameters() if p.requires_grad]

    def print_trainable_params(self):
        total   = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.trainable_parameters())
        print(f"  전체 파라미터:    {total:,} ({total/1e6:.1f}M)")
        print(f"  학습 파라미터:    {trainable:,} ({trainable/1e6:.2f}M)")
        print(f"  학습 비율:        {100*trainable/total:.2f}%")
'''

with open(f'{REV4_ROOT}/models/vla_model_llm.py', 'w') as f:
    f.write(code)
print('✅ [LLM] models/vla_model_llm.py 작성 완료')

✅ [LLM] models/vla_model_llm.py 작성 완료


In [ ]:
# ─── CELL 13: LLM 모델 구조 검증 (CPU, 0.5B로 테스트) ────
# 실제 학습은 A100에서 1.5B 사용 / 여기선 구조 확인만
import torch, sys
sys.path.insert(0, REV4_ROOT)

TEST_LLM = "Qwen/Qwen2.5-0.5B"  # CPU 검증용 (실제 학습은 1.5B)
print(f'모델 로드 중: {TEST_LLM} (CPU 검증용)...')

from models.vla_model_llm import VLAModelLLM

model = VLAModelLLM(
    v_dim=640, n_assets=5,
    llm_name=TEST_LLM,
    device="cpu", use_lora=True, lora_r=16,
)

# 더미 입력
B = 2
v_emb      = torch.randn(B, 640)
news_texts = [
    "Bitcoin surges as institutional demand rises",
    "Crypto market faces regulatory pressure from SEC",
]

print('\n포워드 패스 테스트...')
with torch.no_grad():
    weights, hidden = model(v_emb, news_texts)

print(f'\n✅ [LLM] VLAModelLLM 검증 완료')
print(f'   입력  V:      {v_emb.shape}')
print(f'   입력  뉴스:   {len(news_texts)}개 텍스트')
print(f'   출력 weights: {weights.shape}  (합={weights[0].sum().item():.4f})')
print(f'   출력 hidden:  {hidden.shape}')
print()
model.print_trainable_params()
print()
print('학습 시 1.5B 사용:')
print('  llm_name = "Qwen/Qwen2.5-1.5B"  # vla_model_llm.py 기본값')

In [ ]:
# ─── CELL 14: config.py 작성 ─────────────────────────────
code = '''# X-MultiVLA Rev4 Configuration

# ── 경로 ────────────────────────────────────────────────
REV3_ROOT  = "/content/drive/MyDrive/X-MultiVLA_rev2_5Simbols"
REV4_ROOT  = "/content/drive/MyDrive/X-MultiVLA_rev4"
DATA_DIR   = f"{REV3_ROOT}/data"       # Rev3 데이터 공유
CKPT_DIR   = f"{REV4_ROOT}/checkpoints"
OUT_DIR    = f"{REV4_ROOT}/outputs"
LOG_DIR    = f"{REV4_ROOT}/logs"

# ── 자산 ────────────────────────────────────────────────
ASSETS  = ["BTC", "ETH", "SOL", "XRP", "DOGE"]
SYM_MAP = {
    "BTCUSDT": "BTC", "ETHUSDT": "ETH",
    "SOLUSDT": "SOL", "XRPUSDT": "XRP", "DOGEUSDT": "DOGE",
}
N_ASSETS = len(ASSETS)

# ── PatchTST (V 인코더) ──────────────────────────────────
WINDOW_BARS     = 60    # 20일 (Rev3 30봉 → 60봉으로 확장)
WARMUP_BARS     = 360
PATCH_LEN       = 4
D_MODEL         = 128
N_HEADS         = 4
N_LAYERS        = 3
DROPOUT         = 0.1
EMBED_DIM       = N_ASSETS * D_MODEL  # 640

# ── 사전학습 ─────────────────────────────────────────────
PRETRAIN_EPOCHS = 100
PRETRAIN_LR     = 3e-4
BATCH_SIZE      = 64
DIR_LOSS_WEIGHT = 0.3   # MSE 0.7 + 방향분류 0.3

# ── LLM (L 컴포넌트) ─────────────────────────────────────
LLM_NAME        = "Qwen/Qwen2.5-1.5B"
LLM_NAME_TEST   = "Qwen/Qwen2.5-0.5B"   # CPU 검증용
LORA_R          = 16
LORA_ALPHA      = 32
LLM_MAX_NEWS    = 3      # 버킷당 최대 뉴스 헤드라인 수
LLM_MAX_LENGTH  = 256    # 토크나이저 최대 길이

# ── VLA 융합 ─────────────────────────────────────────────
D_MODEL_FUSION  = 256    # Cross-Attention 내부 차원
N_HEADS_FUSION  = 8

# ── PPO (A 컴포넌트) ─────────────────────────────────────
PPO_TIMESTEPS   = 1_000_000
PPO_LR          = 3e-4

# ── 보상함수 (BTC 벤치마크) ──────────────────────────────
COMMISSION      = 0.001
# reward = port_ret - btc_ret - cost
# 상승장: BTC 초과시 양수 / 하락장: 현금보유시 강한 양수

# ── Walk-Forward ─────────────────────────────────────────
TRAIN_START = "2024-03-01"
ROUNDS = [
    {"name": "R1", "test_start": "2025-03-01", "test_end": "2025-06-01"},
    {"name": "R2", "test_start": "2025-06-01", "test_end": "2025-09-01"},
    {"name": "R3", "test_start": "2025-09-01", "test_end": "2025-12-01"},
    {"name": "R4", "test_start": "2025-12-01", "test_end": "2026-03-01"},
    {"name": "R5", "test_start": "2026-03-01", "test_end": "2026-04-13"},
]

# ── 사용자 Navigation (기본값: 제약 없음) ────────────────
DEFAULT_NAV = {
    "min_cash":   0.0,   # 최소 현금 비중 (0 = 제약 없음)
    "max_single": 1.0,   # 단일 자산 최대 비중 (1 = 제약 없음)
}
'''

with open(f'{REV4_ROOT}/config.py', 'w') as f:
    f.write(code)
print('✅ config.py 작성 완료')
print('   주요 변경: WINDOW_BARS=60, BTC벤치마크 보상, LLM=Qwen2.5-1.5B')

In [ ]:
# ─── CELL 11: config.py 작성 ─────────────────────────────
code = '''# X-MultiVLA Rev4 Configuration

REV3_ROOT = "/content/drive/MyDrive/X-MultiVLA_rev2_5Simbols"
REV4_ROOT = "/content/drive/MyDrive/X-MultiVLA_rev4"
DATA_DIR  = f"{REV4_ROOT}/data"
SRC_DIR   = f"{REV4_ROOT}/src"
CKPT_DIR  = f"{REV4_ROOT}/checkpoints"
OUT_DIR   = f"{REV4_ROOT}/outputs"
LOG_DIR   = f"{REV4_ROOT}/logs"

DATASET_FILE  = f"{DATA_DIR}/dataset_8hr_full.parquet"
BINANCE_FILE  = f"{DATA_DIR}/binance_processed_futures_8hr_260516_fixed.parquet"
NEWS_FILE     = f"{DATA_DIR}/news_all_8h.csv"
SENTIMENT_FILE= f"{DATA_DIR}/news_sentiment_8h.csv"  # FinBERT 결과 (재계산 불필요)

ASSETS   = ["BTC", "ETH", "SOL", "XRP", "DOGE"]
SYM_MAP  = {"BTCUSDT":"BTC","ETHUSDT":"ETH","SOLUSDT":"SOL",
            "XRPUSDT":"XRP","DOGEUSDT":"DOGE"}
N_ASSETS = len(ASSETS)

NEWS_COLS = ["news_count","news_sentiment_mean","news_pos_mean","news_neg_mean"]
META_COLS = ["symbol","open_time","date","target","close","symbol_encoded"]
N_FEATURES = 106

# iTransformer
SEQ_LEN  = 60
D_MODEL  = 256
N_HEADS  = 8
N_LAYERS = 4
DROPOUT  = 0.1

# L 컴포넌트: news_sentiment_8h.csv 기반 (pos, neg, score × 5코인)
L_DIM    = 3    # FinBERT 분류 헤드 출력 (재계산 불필요)
LLM_MAX_NEWS = 3

# LLM
LLM_NAME      = "Qwen/Qwen2.5-1.5B"
LLM_NAME_TEST = "Qwen/Qwen2.5-0.5B"
LORA_R        = 16
LORA_ALPHA    = 32
LLM_MAX_LEN   = 256

# Phase 1
ORACLE_EPOCHS    = 30
ORACLE_LR        = 3e-4
BATCH_SIZE       = 32
FOCAL_WEIGHT     = 10.0
FOCAL_PERCENTILE = 95.0

# Phase 2 GRPO
GRPO_STEPS          = 30_000
GRPO_GROUP_SIZE     = 8
GRPO_LR             = 1e-4
GRPO_KL_BETA        = 0.01
GRPO_CLIP_EPS       = 0.2
GRPO_DIRICHLET_CONC = 10.0
SORTINO_LAMBDA      = 0.1

COMMISSION = 0.001

TRAIN_START = "2024-03-01"
ROUNDS = [
    {"name":"R1","test_start":"2025-03-01","test_end":"2025-06-01"},
    {"name":"R2","test_start":"2025-06-01","test_end":"2025-09-01"},
    {"name":"R3","test_start":"2025-09-01","test_end":"2025-12-01"},
    {"name":"R4","test_start":"2025-12-01","test_end":"2026-03-01"},
    {"name":"R5","test_start":"2026-03-01","test_end":"2026-04-13"},
]
DEFAULT_NAV = {"min_cash": 0.0, "max_single": 1.0}
'''
with open(f'{REV4_ROOT}/config.py', 'w') as f:
    f.write(code)
print('✅ config.py — L_DIM=3 (기존 FinBERT 감성점수 재활용)')

✅ config.py 업데이트
   N_FEATURES = 106 (실측값 반영)


In [ ]:
# ─── CELL 9: src/training/oracle_labels.py 작성 ──────────
code = '''import numpy as np
import torch
import torch.nn.functional as F

class OracleLabelGenerator:
    """
    Phase 1 지도학습 정답 레이블 생성기

    설계:
    - Backward DP: 전체 구간 전역 최적 (룩어헤드 고정값 없음)
    - 경로 패널티: effective_port_ret = raw_port_ret - vol_weight * intraday_mdd
      → R[t][s][a] = effective_port_ret - btc_ret - switching_cost (중복 차감 없음)
    - Focal Loss 가중치: 상위 5% 극단 변동 구간 10배 가중치
    - 소프트 레이블: Q값 비례 분산
    """
    def __init__(self, commission: float = 0.001, n_assets: int = 5,
                 gamma: float = 0.99,
                 label_type: str = "soft",
                 temperature: float = 5.0,
                 volatility_penalty: float = 0.5,
                 focal_weight: float = 10.0,
                 focal_percentile: float = 95.0):
        self.comm             = commission
        self.n_assets         = n_assets
        self.gamma            = gamma
        self.label_type       = label_type
        self.temperature      = temperature
        self.vol_penalty      = volatility_penalty
        self.focal_weight     = focal_weight      # 롱테일 구간 Loss 가중치
        self.focal_percentile = focal_percentile  # 상위 몇 % 구간에 적용

        self.n_actions       = n_assets + 1
        self._action_weights = np.array(
            [self._one_hot(i) for i in range(n_assets)] + [self._cash()],
            dtype=np.float32)

    def _cash(self):
        w = np.zeros(self.n_assets + 1, dtype=np.float32); w[-1] = 1.0; return w

    def _one_hot(self, idx):
        w = np.zeros(self.n_assets + 1, dtype=np.float32); w[idx] = 1.0; return w

    def _reward_matrix(self, prices, t, high_prices=None, low_prices=None):
        """
        R[s][a] = effective_port_ret - btc_ret - switching_cost
        effective_port_ret = raw_port_ret - vol_penalty * intraday_mdd
        (path_penalty는 effective_port_ret에 이미 포함 → 중복 차감 없음)
        """
        rets    = (prices[t+1] - prices[t]) / (prices[t] + 1e-9)
        btc_ret = rets[0]

        if high_prices is not None and low_prices is not None:
            intraday_mdd = (high_prices[t] - low_prices[t]) / (prices[t] + 1e-9)
        else:
            intraday_mdd = np.abs(rets)

        R = np.zeros((self.n_actions, self.n_actions), dtype=np.float32)
        for s in range(self.n_actions):
            w_prev = self._action_weights[s]
            for a in range(self.n_actions):
                w_new    = self._action_weights[a]
                cost     = self.comm * np.abs(w_new - w_prev).sum()
                raw_ret  = float(np.dot(w_new[:self.n_assets], rets))

                # 경로 패널티: effective_port_ret으로 통합 (중복 차감 방지)
                if a < self.n_assets:
                    effective_ret = raw_ret - self.vol_penalty * float(intraday_mdd[a])
                else:
                    effective_ret = raw_ret  # 현금은 패널티 없음

                R[s, a] = effective_ret - btc_ret - cost
        return R

    def compute_focal_weights(self, high_prices, low_prices, prices):
        """
        각 타임스텝의 Focal Loss 가중치 계산
        intraday_mdd 상위 focal_percentile% 구간 → focal_weight 배
        """
        T = len(prices) - 1
        if high_prices is None or low_prices is None:
            return np.ones(T, dtype=np.float32)

        mdd_mean = np.mean(
            (high_prices[:T] - low_prices[:T]) / (prices[:T] + 1e-9), axis=1)
        threshold = np.percentile(mdd_mean, self.focal_percentile)

        weights = np.where(mdd_mean >= threshold,
                           self.focal_weight, 1.0).astype(np.float32)
        n_focal = (weights > 1.0).sum()
        return weights  # (T,)

    def generate(self, prices, high_prices=None, low_prices=None,
                 init_action=None, verbose=True):
        """
        전체 구간 Backward DP → 전역 최적 레이블 + Focal 가중치

        반환:
          labels:        (T, n_assets+1) 정답 포트폴리오 비중
          focal_weights: (T,)            Loss 가중치 (롱테일 구간 focal_weight배)
        """
        T  = len(prices) - 1
        A  = self.n_actions
        s0 = init_action if init_action is not None else self.n_assets

        if verbose:
            print(f'  Oracle DP: 보상 행렬 계산 중... T={T}')

        R_all = np.zeros((T, A, A), dtype=np.float32)
        for t in range(T):
            R_all[t] = self._reward_matrix(prices, t, high_prices, low_prices)

        # Backward DP
        V      = np.zeros(A, dtype=np.float32)
        policy = np.zeros((T, A), dtype=np.int32)
        Q_all  = np.zeros((T, A, A), dtype=np.float32)

        for t in range(T - 1, -1, -1):
            Q = R_all[t] + self.gamma * V[np.newaxis, :]
            Q_all[t]  = Q
            policy[t] = Q.argmax(axis=1)
            V         = Q.max(axis=1)

        # Forward 패스
        labels    = np.zeros((T, self.n_assets + 1), dtype=np.float32)
        cur_state = s0

        for t in range(T):
            best_a = policy[t, cur_state]
            if self.label_type == "soft":
                q_vals = Q_all[t, cur_state]
                probs  = F.softmax(
                    torch.tensor(q_vals * self.temperature), dim=0).numpy()
                label  = (probs[:, np.newaxis] * self._action_weights).sum(axis=0)
                label /= label.sum() + 1e-8
            else:
                label = self._action_weights[best_a].copy()
            labels[t]  = label
            cur_state  = best_a

        # Focal 가중치
        focal_weights = self.compute_focal_weights(high_prices, low_prices, prices)

        if verbose:
            cash_r   = (labels[:, -1] > 0.5).mean() * 100
            n_focal  = (focal_weights > 1.0).sum()
            print(f"  완료: 현금보유 {cash_r:.1f}% | "
                  f"Focal 구간 {n_focal}/{T} ({n_focal/T*100:.1f}%, "
                  f"가중치 {self.focal_weight}x)")

        return labels, focal_weights
'''
with open(f'{REV4_SRC}/training/oracle_labels.py', 'w') as f:
    f.write(code)
open(f'{REV4_SRC}/training/__init__.py', 'w').close()
print('✅ src/training/oracle_labels.py')
print('   Focal Loss: 상위 5% 극단 변동 구간 10x 가중치')
print('   중복 차감 수정: effective_port_ret으로 통합')

✅ src/training/oracle_labels.py
   Focal Loss: 상위 5% 극단 변동 구간 10x 가중치
   중복 차감 수정: effective_port_ret으로 통합


In [ ]:
# ─── CELL 10: src/training/grpo_trainer.py 작성 ──────────
code = '''import torch, torch.nn as nn, copy, numpy as np
from torch.distributions import Dirichlet
from tqdm import tqdm

class GRPOTrainer:
    """Phase 2 GRPO — l_embs: (T, 5, 768) 시퀀스 입력"""
    def __init__(self, model, config, device="cuda"):
        self.model  = model
        self.cfg    = config
        self.device = device
        self.opt    = torch.optim.AdamW(
            model.trainable_parameters(), lr=config.GRPO_LR, weight_decay=1e-4)
        self.ref_model       = None
        self._return_history = []

    def _init_ref(self):
        self.ref_model = copy.deepcopy(self.model)
        for p in self.ref_model.parameters(): p.requires_grad_(False)
        self.ref_model.eval()

    def _downside_vol(self, window=50):
        if len(self._return_history) < 5: return 0.0
        r = np.array(self._return_history[-window:])
        d = r[r < 0]
        return float(np.std(d)) if len(d) > 1 else 0.0

    def _rewards(self, samples, prices, t, prev_w):
        rets    = (prices[t+1]-prices[t])/(prices[t]+1e-9)
        btc_ret = float(rets[0])
        lam     = getattr(self.cfg,'SORTINO_LAMBDA',0.1)
        dv      = self._downside_vol()
        rews    = []
        for w in samples.cpu().numpy():
            port_ret = float(np.dot(w[:len(rets)],rets))
            cost     = self.cfg.COMMISSION * np.abs(w-prev_w).sum()
            rews.append(port_ret-btc_ret-cost-lam*dv)
            self._return_history.append(port_ret-btc_ret)
        return torch.tensor(rews, dtype=torch.float32, device=self.device)

    def _loss(self, lp, ref_lp, rewards):
        adv     = (rewards-rewards.mean())/(rewards.std()+1e-8)
        ratio   = torch.exp(lp-ref_lp)
        clipped = ratio.clamp(1-self.cfg.GRPO_CLIP_EPS, 1+self.cfg.GRPO_CLIP_EPS)
        return (-torch.min(ratio*adv, clipped*adv).mean()
                + self.cfg.GRPO_KL_BETA*(lp-ref_lp).mean())

    def train(self, v_embs, l_embs, news_texts, prices, n_steps=None, rname=""):
        """
        v_embs:     (T, d_model)   — iTransformer 사전계산
        l_embs:     (T, 5, 768)    — 코인별 FinBERT (5토큰)
        news_texts: List[str] T개
        prices:     (T+1, n_assets)
        """
        if self.ref_model is None: self._init_ref()
        n_steps=n_steps or self.cfg.GRPO_STEPS
        T,G   = len(v_embs)-1, self.cfg.GRPO_GROUP_SIZE
        prev_w= np.zeros(prices.shape[1]+1); prev_w[-1]=1.0
        total = 0.0; self._return_history=[]

        self.model.train()
        pbar=tqdm(range(n_steps), desc=f"GRPO {rname}")
        for step in pbar:
            t    = np.random.randint(0,T)
            v_t  = torch.FloatTensor(v_embs[t]).unsqueeze(0).to(self.device)
            l_t  = torch.FloatTensor(l_embs[t]).unsqueeze(0).to(self.device)  # (1,5,768)
            text = [news_texts[t]]

            w,_,_ = self.model.forward_from_v_emb(v_t,l_t,text)
            w=w.squeeze(0)
            conc  = (w*self.cfg.GRPO_DIRICHLET_CONC).clamp(min=0.1)
            dist  = Dirichlet(conc)
            samp  = dist.sample((G,))
            lp    = dist.log_prob(samp)

            with torch.no_grad():
                rw,_,_=self.ref_model.forward_from_v_emb(v_t,l_t,text)
                rc   =(rw.squeeze(0)*self.cfg.GRPO_DIRICHLET_CONC).clamp(min=0.1)
                ref_lp=Dirichlet(rc).log_prob(samp)

            rew  = self._rewards(samp,prices,t,prev_w)
            loss = self._loss(lp,ref_lp,rew)

            self.opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(self.model.trainable_parameters(),1.0)
            self.opt.step()
            prev_w=w.detach().cpu().numpy(); total+=loss.item()
            if (step+1)%1000==0:
                pbar.set_postfix({"loss":f"{total/(step+1):.4f}",
                                  "rew":f"{rew.mean().item():.4f}"})
        return total/n_steps
'''
with open(f'{REV4_SRC}/training/grpo_trainer.py', 'w') as f:
    f.write(code)
print('✅ src/training/grpo_trainer.py — l_embs (T,5,768) 수정')

✅ src/training/grpo_trainer.py — l_embs (T,5,768) 수정


In [ ]:
# ─── CELL 10: src/utils/news_processor.py 작성 ───────────
code = '''import pandas as pd
import numpy as np

class NewsProcessor:
    """뉴스 CSV → 타임스텝별 텍스트 + FinBERT 임베딩 집계"""
    def __init__(self, data_dir: str, max_headlines: int = 3):
        self.data_dir      = data_dir
        self.max_headlines = max_headlines

    def load_precomputed(self) -> dict:
        """news_finbert_768d.pkl 로드 (사전계산된 임베딩)"""
        import joblib, os
        path = f"{self.data_dir}/news_finbert_768d.pkl"
        if not os.path.exists(path):
            raise FileNotFoundError(f"사전계산 임베딩 없음: {path}\\nCELL 21 먼저 실행")
        return joblib.load(path)

    def build_embedding_map(self, assets: list,
                             timestamps: pd.DatetimeIndex) -> dict:
        """
        사전계산된 768d 임베딩을 타임스텝별로 집계
        반환: {coin: np.ndarray (T, 768)}
        """
        data  = self.load_precomputed()
        embs  = data["embeddings"]     # (N, 768)
        coins = data["coin"]
        times = pd.DatetimeIndex(data["datetime_8h"])
        result = {}

        for coin in assets:
            mask = coins == coin
            coin_embs  = embs[mask]
            coin_times = times[mask]
            out = []
            for ts in timestamps:
                idx = np.where(coin_times == ts)[0]
                if len(idx):
                    out.append(coin_embs[idx].mean(axis=0))
                else:
                    out.append(np.zeros(768, dtype=np.float32))
            result[coin] = np.stack(out)   # (T, 768)
        return result

    def build_text_map(self, assets: list,
                        timestamps: pd.DatetimeIndex) -> dict:
        """LLM 입력용 텍스트 맵 반환: {coin: [text_t0, ...]}"""
        df = pd.read_csv(f"{self.data_dir}/news_sentiment_8h.csv",
                         parse_dates=["published_at"], encoding="utf-8")
        df["datetime_8h"] = pd.to_datetime(df["published_at"]).dt.floor("8h")
        result = {}
        for coin in assets:
            sub   = df[df["coin"] == coin]
            texts = []
            for ts in timestamps:
                bucket = sub[sub["datetime_8h"] == ts].nlargest(
                    self.max_headlines, "score")["title"].tolist()
                texts.append(" | ".join(str(h) for h in bucket)
                             if bucket else f"No significant {coin} news.")
            result[coin] = texts
        return result

    def build_combined_text(self, text_map: dict, assets: list, t: int) -> str:
        return " ".join(f"[{c}] {text_map[c][t]}" for c in assets)
'''
with open(f'{REV4_SRC}/utils/news_processor.py', 'w') as f:
    f.write(code)
open(f'{REV4_SRC}/utils/__init__.py', 'w').close()
print('✅ src/utils/news_processor.py')

✅ src/utils/news_processor.py


In [ ]:
# ─── CELL 13: 전체 파일 구조 확인 ────────────────────────
import os

print('='*58)
print('  X-MultiVLA Rev4 — MD 기준 파일 구조')
print('='*58)

files = [
    ('config.py',                        'Rev4 설정 (GRPO, LLM, iTransformer)'),
    ('src/models/itransformer.py',       '★ V: iTransformer (코인 간 어텐션)'),
    ('src/models/cross_attention.py',    '★ L: Cross-Attention (V×FinBERT)'),
    ('src/models/llm_reasoning.py',      '★ L: LLM (Qwen2.5-1.5B + LoRA)'),
    ('src/models/action_head.py',        '★ A: Action Head + Navigation'),
    ('src/models/vla_model.py',          '★ E2E VLA 통합 모델'),
    ('src/models/finbert_encoder.py',    '  FinBERT 768d 인코더'),
    ('src/training/oracle_labels.py',    '★ Phase 1: Oracle 정답 레이블'),
    ('src/training/grpo_trainer.py',     '★ Phase 2: GRPO Fine-tuning'),
    ('src/utils/news_processor.py',      '  뉴스 텍스트/임베딩 처리'),
    ('src/utils/evaluator.py',           '  백테스트 평가'),
    ('src/data/collector.py',            '  데이터 수집'),
    ('src/data/feature_engineer.py',     '  피처 엔지니어링'),
    ('src/data/preprocessor.py',         '  전처리'),
]

all_ok = True
for fname, desc in files:
    path   = f'{REV4_ROOT}/{fname}'
    exists = os.path.exists(path)
    mark   = '✅' if exists else '❌'
    print(f'  {mark}  {fname:<38} {desc}')
    if not exists: all_ok = False

print('='*58)
print('  ✅ 완료!' if all_ok else '  ❌ 누락 있음 — 위 셀 재실행 필요')
print()
print('학습 흐름 (MD 기준):')
print('  Phase 1: Oracle SL  → iTransformer+LLM+ActionHead 기본 학습')
print('  Phase 2: GRPO       → 수수료+순서의존성 최적화')
print('  Walk-Forward 5라운드 × 병렬 실행')

  X-MultiVLA Rev4 — MD 기준 파일 구조
  ✅  config.py                              Rev4 설정 (GRPO, LLM, iTransformer)
  ✅  src/models/itransformer.py             ★ V: iTransformer (코인 간 어텐션)
  ✅  src/models/cross_attention.py          ★ L: Cross-Attention (V×FinBERT)
  ✅  src/models/llm_reasoning.py            ★ L: LLM (Qwen2.5-1.5B + LoRA)
  ✅  src/models/action_head.py              ★ A: Action Head + Navigation
  ✅  src/models/vla_model.py                ★ E2E VLA 통합 모델
  ✅  src/models/finbert_encoder.py            FinBERT 768d 인코더
  ✅  src/training/oracle_labels.py          ★ Phase 1: Oracle 정답 레이블
  ✅  src/training/grpo_trainer.py           ★ Phase 2: GRPO Fine-tuning
  ✅  src/utils/news_processor.py              뉴스 텍스트/임베딩 처리
  ✅  src/utils/evaluator.py                   백테스트 평가
  ✅  src/data/collector.py                    데이터 수집
  ✅  src/data/feature_engineer.py             피처 엔지니어링
  ✅  src/data/preprocessor.py                 전처리
  ✅ 완료!

학습 흐름 (MD 기준):
  Phase 1: Oracle SL  → iTransf

In [ ]:
# ─── 데이터 폴더 확인 ────────────────────────────────────
import os

print('Rev4 data/ 폴더 파일 목록:')
for f in sorted(os.listdir(REV4_DATA)):
    size = os.path.getsize(f'{REV4_DATA}/{f}') / 1024 / 1024
    print(f'  {f:<50} {size:.1f} MB')

Rev4 data/ 폴더 파일 목록:
  binance_processed_futures_8hr_260516_fixed.parquet 1.1 MB
  dataset_8hr_full.parquet                           4.8 MB
  multi_asset_preprocessor.py                        0.0 MB
  news_all_8h.csv                                    83.5 MB
  news_sentiment_8h.csv                              43.0 MB


In [ ]:
# ─── 데이터 구조 확인 + 정리 ─────────────────────────────
import pandas as pd
import os

# 코드 파일 제거
stale = f'{REV4_DATA}/multi_asset_preprocessor.py'
if os.path.exists(stale):
    os.remove(stale)
    print('🗑️  data/multi_asset_preprocessor.py 제거\n')

# Binance 선물 데이터
print('='*55)
print('1. binance_processed_futures_8hr_260516_fixed.parquet')
print('='*55)
df_bin = pd.read_parquet(f'{REV4_DATA}/binance_processed_futures_8hr_260516_fixed.parquet')
print(f'shape: {df_bin.shape}')
print(f'columns: {df_bin.columns.tolist()}')
if 'Open time' in df_bin.columns:
    df_bin['Open time'] = pd.to_datetime(df_bin['Open time'])
    print(f'기간: {df_bin["Open time"].min().date()} ~ {df_bin["Open time"].max().date()}')
if 'Symbol' in df_bin.columns:
    print(f'심볼: {df_bin["Symbol"].unique()}')
print()

# Full dataset
print('='*55)
print('2. dataset_8hr_full.parquet')
print('='*55)
df_full = pd.read_parquet(f'{REV4_DATA}/dataset_8hr_full.parquet')
print(f'shape: {df_full.shape}')
print(f'columns (처음 20개): {df_full.columns.tolist()[:20]}')
print(f'index: {df_full.index[:3]}')
print()

# 뉴스 데이터
print('='*55)
print('3. news_all_8h.csv (원본)')
print('='*55)
df_news = pd.read_csv(f'{REV4_DATA}/news_all_8h.csv', nrows=3)
print(f'columns: {df_news.columns.tolist()}')
print(df_news.head(2))

🗑️  data/multi_asset_preprocessor.py 제거

1. binance_processed_futures_8hr_260516_fixed.parquet
shape: (11585, 13)
columns: ['Open', 'High', 'Low', 'Close', 'Volume', 'Close time', 'Quote asset volume', 'Number of trades', 'Taker buy base asset volume', 'Taker buy quote asset volume', 'Symbol', 'Open time', 'fundingRate']
기간: 2024-03-01 ~ 2026-04-12
심볼: ['BTCUSDT' 'ETHUSDT' 'DOGEUSDT' 'XRPUSDT' 'SOLUSDT']

2. dataset_8hr_full.parquet
shape: (11580, 116)
columns (처음 20개): ['symbol', 'open_time', 'date', 'target', 'open', 'close', 'funding_rate', 'rsi_7', 'rsi_14', 'rsi_21', 'macd_hist_norm', 'bb_pct', 'bb_width', 'stoch_k', 'stoch_d', 'ema_5_ratio', 'ema_10_ratio', 'ema_21_ratio', 'ema_55_ratio', 'ema_90_ratio']
index: RangeIndex(start=0, stop=3, step=1)

3. news_all_8h.csv (원본)
columns: ['coin', 'title', 'published_at', 'url', 'source', 'date', 'bucket_8h']
   coin                                              title  \
0  DOGE  Bitcoin Volatility Induces $700 Million Carnag...   
1   BTC

In [ ]:
# ─── dataset_8hr_full 상세 확인 ──────────────────────────
import pandas as pd
import numpy as np

df = pd.read_parquet(f'{REV4_DATA}/dataset_8hr_full.parquet')

print(f'전체 shape: {df.shape}')
print(f'심볼: {df["symbol"].unique()}')
print(f'기간: {df["open_time"].min()} ~ {df["open_time"].max()}')
print(f'\ntarget 컬럼 샘플:')
print(df[["symbol","open_time","target","close"]].head(6))
print(f'\ntarget 통계:')
print(df["target"].describe())
print(f'\n전체 컬럼 ({df.shape[1]}개):')
print(df.columns.tolist())

전체 shape: (11580, 116)
심볼: ['BTCUSDT' 'DOGEUSDT' 'ETHUSDT' 'SOLUSDT' 'XRPUSDT']
기간: 2024-03-01 00:00:00 ~ 2026-04-11 16:00:00

target 컬럼 샘플:
    symbol           open_time    target         close
0  BTCUSDT 2024-03-01 00:00:00  0.006572  61575.300781
1  BTCUSDT 2024-03-01 08:00:00 -0.007906  61980.000000
2  BTCUSDT 2024-03-01 16:00:00  0.014329  61490.000000
3  BTCUSDT 2024-03-02 00:00:00 -0.005520  62371.101562
4  BTCUSDT 2024-03-02 08:00:00 -0.001514  62026.800781
5  BTCUSDT 2024-03-02 16:00:00 -0.002162  61932.898438

target 통계:
count    11580.000000
mean         0.000268
std          0.023628
min         -0.174211
25%         -0.010238
50%          0.000080
75%          0.010573
max          0.267821
Name: target, dtype: float64

전체 컬럼 (116개):
['symbol', 'open_time', 'date', 'target', 'open', 'close', 'funding_rate', 'rsi_7', 'rsi_14', 'rsi_21', 'macd_hist_norm', 'bb_pct', 'bb_width', 'stoch_k', 'stoch_d', 'ema_5_ratio', 'ema_10_ratio', 'ema_21_ratio', 'ema_55_ratio', 'ema_90_ratio

In [ ]:
# ─── 뉴스-코인 매칭 구조 확인 ────────────────────────────
import pandas as pd

df_news = pd.read_csv(f'{REV4_DATA}/news_all_8h.csv', parse_dates=['published_at'])
df_data = pd.read_parquet(f'{REV4_DATA}/dataset_8hr_full.parquet')

print('=== 뉴스 데이터 ===')
print(f'coin 종류: {df_news["coin"].unique()}')
print(f'bucket_8h 예시: {df_news["bucket_8h"].head(3).tolist()}')
print(f'published_at 예시: {df_news["published_at"].head(3).tolist()}')

print('\n=== dataset 코인/시간 ===')
print(f'symbol 종류: {df_data["symbol"].unique()}')
print(f'open_time 예시: {df_data["open_time"].head(3).tolist()}')

# 매칭 테스트: BTC 2024-03-01 00:00 기준
ts = pd.Timestamp('2024-03-01 00:00:00')
news_btc = df_news[
    (df_news['coin'] == 'BTC') &
    (pd.to_datetime(df_news['published_at']).dt.floor('8h') == ts)
]
print(f'\nBTC + 2024-03-01 00:00 뉴스 건수: {len(news_btc)}')
print(news_btc[['coin','title','published_at']].head(3))

# 각 코인별 뉴스 건수
print('\n코인별 전체 뉴스 건수:')
print(df_news['coin'].value_counts())

=== 뉴스 데이터 ===
coin 종류: ['DOGE' 'BTC' 'SOL' 'ETH' 'XRP']
bucket_8h 예시: ['00h-08h', '00h-08h', '00h-08h']
published_at 예시: [Timestamp('2024-03-01 00:00:36'), Timestamp('2024-03-01 00:00:36'), Timestamp('2024-03-01 00:06:04')]

=== dataset 코인/시간 ===
symbol 종류: ['BTCUSDT' 'DOGEUSDT' 'ETHUSDT' 'SOLUSDT' 'XRPUSDT']
open_time 예시: [Timestamp('2024-03-01 00:00:00'), Timestamp('2024-03-01 08:00:00'), Timestamp('2024-03-01 16:00:00')]

BTC + 2024-03-01 00:00 뉴스 건수: 52
  coin                                              title        published_at
1  BTC  Bitcoin Volatility Induces $700 Million Carnag... 2024-03-01 00:00:36
3  BTC  Bitcoin Billionaires: The Winklevoss Twins' Li... 2024-03-01 00:06:04
7  BTC  Solana (SOL) Performs Enormous Breakthrough, E... 2024-03-01 00:30:00

코인별 전체 뉴스 건수:
coin
BTC     186580
ETH      78115
XRP      51638
SOL      40558
DOGE     25357
Name: count, dtype: int64


In [ ]:
# ─── CELL TRAIN-1: 데이터 로딩 + 윈도우 생성 ─────────────
import pandas as pd
import numpy as np
import sys, os
sys.path.insert(0, REV4_SRC)
sys.path.insert(0, REV4_ROOT)

import importlib, config
importlib.reload(config)
from config import *

# ── 데이터 로드 ───────────────────────────────────────────
df = pd.read_parquet(DATASET_FILE)
df['open_time'] = pd.to_datetime(df['open_time'])
df_bin = pd.read_parquet(BINANCE_FILE)
df_bin['Open time'] = pd.to_datetime(df_bin['Open time'])

print(f'Dataset: {df.shape}  기간: {df.open_time.min().date()} ~ {df.open_time.max().date()}')

# ── V 피처 컬럼 (뉴스 감성 제외) ─────────────────────────
EXCLUDE_COLS = set(NEWS_COLS + META_COLS)
FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE_COLS]
N_FEATURES   = len(FEATURE_COLS)
print(f'V 피처: {N_FEATURES}개 (뉴스 감성 {len(NEWS_COLS)}개 제외)')

# ── 코인별 DataFrame 분리 ─────────────────────────────────
coin_dfs = {}
for sym, coin in SYM_MAP.items():
    sub = df[df['symbol'] == sym].sort_values('open_time').reset_index(drop=True)
    sub.index = sub['open_time']
    coin_dfs[coin] = sub

timestamps = sorted(set.intersection(*[set(d.index) for d in coin_dfs.values()]))
timestamps = pd.DatetimeIndex(timestamps)
print(f'공통 타임스텝: {len(timestamps)}개')

# ── 가격 배열 (Oracle + 보상 계산용) ─────────────────────
price_arr = np.stack([
    coin_dfs[c]['close'].reindex(timestamps).values for c in ASSETS
], axis=1)  # (T, 5)
print(f'price_arr: {price_arr.shape}')

# ── V 피처 배열 (T, N, F) ─────────────────────────────────
feat_arr = np.stack([
    coin_dfs[c][FEATURE_COLS].reindex(timestamps).fillna(0).values
    for c in ASSETS
], axis=1).astype(np.float32)  # (T, N, F)
print(f'feat_arr: {feat_arr.shape}  (T, N={N_ASSETS}, F={N_FEATURES})')

# ── 슬라이딩 윈도우 생성 ──────────────────────────────────
def make_windows(arr, seq_len):
    """(T, N, F) → (T-seq_len+1, seq_len, N, F)"""
    T = len(arr)
    return np.stack([arr[i:i+seq_len] for i in range(T - seq_len + 1)])

windows = make_windows(feat_arr, SEQ_LEN)     # (W, 60, 5, F)
prices_w = price_arr[SEQ_LEN-1:]              # (W+1, 5) aligned
print(f'윈도우: {windows.shape}  가격: {prices_w.shape}')
print(f'\n✅ 데이터 로딩 완료')

Dataset: (11580, 116)  기간: 2024-03-01 ~ 2026-04-11
V 피처: 106개 (뉴스 감성 4개 제외)
공통 타임스텝: 2316개
price_arr: (2316, 5)
feat_arr: (2316, 5, 106)  (T, N=5, F=106)
윈도우: (2257, 60, 5, 106)  가격: (2257, 5)

✅ 데이터 로딩 완료


In [ ]:
# ─── CELL TRAIN-2: FinBERT 768d 임베딩 사전계산 (1회만) ──
import torch, joblib, os, pandas as pd
from transformers import AutoTokenizer, AutoModel
from google.colab import drive
import importlib, sys, config; importlib.reload(config)
from config import *

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.makedirs(CKPT_DIR, exist_ok=True)

FINBERT_CACHE = f'{CKPT_DIR}/finbert_embeddings.pkl'

if os.path.exists(FINBERT_CACHE):
    print(f'✅ 캐시 이미 존재: {FINBERT_CACHE}')
    cache = joblib.load(FINBERT_CACHE)
    print(f'   저장된 버킷 수: {len(cache)}개')
else:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'FinBERT 768d 사전계산 중... ({device})')

    tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
    model     = AutoModel.from_pretrained('ProsusAI/finbert').to(device)
    for p in model.parameters(): p.requires_grad_(False)
    model.eval()

    df_news = pd.read_csv(NEWS_FILE, parse_dates=['published_at'])
    df_news['ts'] = pd.to_datetime(df_news['published_at']).dt.floor('8h')
    print(f'뉴스 데이터: {len(df_news):,}건')

    cache = {}
    for coin in ASSETS:
        sub = df_news[df_news['coin'] == coin]
        buckets = sub.groupby('ts')
        for ts, group in buckets:
            headlines = group.nlargest(LLM_MAX_NEWS, 'published_at')['title'].tolist()
            if not headlines:
                cache[(coin, ts)] = __import__('numpy').zeros(768, dtype='float32')
                continue
            enc = tokenizer(headlines, padding=True, truncation=True,
                            max_length=128, return_tensors='pt').to(device)
            import torch
            with torch.no_grad():
                out = model(**enc)
            emb = out.last_hidden_state[:,0,:].mean(0).cpu().numpy().astype('float32')
            cache[(coin, ts)] = emb
        print(f'  ✅ {coin}: {len(sub["ts"].unique())}개 버킷')

    joblib.dump(cache, FINBERT_CACHE)
    print(f'\n✅ 저장 완료: {FINBERT_CACHE}')
    print(f'   총 버킷: {len(cache)}개')

FinBERT 768d 사전계산 중... (cuda)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.weight            | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


뉴스 데이터: 382,248건
  ✅ BTC: 2316개 버킷
  ✅ ETH: 2316개 버킷
  ✅ SOL: 2303개 버킷
  ✅ XRP: 2316개 버킷
  ✅ DOGE: 2282개 버킷

✅ 저장 완료: /content/drive/MyDrive/X-MultiVLA_rev4/checkpoints/finbert_embeddings.pkl
   총 버킷: 11533개


In [ ]:
# ─── CELL TRAIN-3: Walk-Forward 학습 (병렬 실행 지원) ────
# ★ 세션마다 RUN_ROUND만 변경
RUN_ROUND = 'R1'

import sys
for k in list(sys.modules.keys()):
    if any(x in k for x in ['vla_model','llm_reasoning','cross_attention',
                              'itransformer','action_head','oracle_labels','grpo_trainer']):
        del sys.modules[k]

import torch, json, joblib, gc, os
import pandas as pd, numpy as np
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from models.vla_model import VLAModel
from training.oracle_labels import OracleLabelGenerator
import importlib, config; importlib.reload(config)
from config import *

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} | {torch.cuda.get_device_name(0) if device=="cuda" else "CPU"}')
os.makedirs(CKPT_DIR, exist_ok=True)

# 이미 완료된 라운드 체크
RESULT_PATH = f'{CKPT_DIR}/wf_results_{RUN_ROUND}.json'
if os.path.exists(RESULT_PATH):
    print(f'✅ {RUN_ROUND} 이미 완료됨'); raise SystemExit

# Drive 마운트
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# ── 데이터 로딩 ───────────────────────────────────────────
df = pd.read_parquet(DATASET_FILE)
df['open_time'] = pd.to_datetime(df['open_time'])
FEATURE_COLS = [c for c in df.columns if c not in set(NEWS_COLS + META_COLS)]

coin_dfs = {}
for sym, coin in SYM_MAP.items():
    sub = df[df['symbol']==sym].sort_values('open_time').reset_index(drop=True)
    sub.index = sub['open_time']
    coin_dfs[coin] = sub

timestamps = pd.DatetimeIndex(sorted(set.intersection(*[set(d.index) for d in coin_dfs.values()])))
price_arr  = np.stack([coin_dfs[c]['close'].reindex(timestamps).values for c in ASSETS], axis=1)
feat_arr   = np.stack([coin_dfs[c][FEATURE_COLS].reindex(timestamps).fillna(0).values
                        for c in ASSETS], axis=1).astype(np.float32)
windows    = np.stack([feat_arr[i:i+SEQ_LEN] for i in range(len(feat_arr)-SEQ_LEN+1)])
prices_w   = price_arr[SEQ_LEN-1:]
win_ts     = timestamps[SEQ_LEN-1:]
print(f'V 피처: {len(FEATURE_COLS)}개 | 윈도우: {len(windows)}개')

# ── L 임베딩: news_sentiment_8h.csv (재계산 불필요) ────────
df_sent = pd.read_csv(SENTIMENT_FILE, parse_dates=['published_at'])
df_sent['ts'] = pd.to_datetime(df_sent['published_at']).dt.floor('8h')

# 코인별 타임스텝별 감성 집계: (T, 5, 3) — pos, neg, score
l_arr = np.zeros((len(timestamps), N_ASSETS, L_DIM), dtype=np.float32)
for j, coin in enumerate(ASSETS):
    sub = df_sent[df_sent['coin'] == coin]
    agg = sub.groupby('ts')[['pos','neg','score']].mean()
    for i, ts in enumerate(timestamps):
        if ts in agg.index:
            l_arr[i, j] = agg.loc[ts, ['pos','neg','score']].values.astype(np.float32)
print(f'L 임베딩: {l_arr.shape} (5코인 × pos/neg/score)')

# ── 라운드 설정 ───────────────────────────────────────────
round_info  = next(r for r in ROUNDS if r['name'] == RUN_ROUND)
test_start  = pd.Timestamp(round_info['test_start'])
test_end    = pd.Timestamp(round_info['test_end'])
train_start = pd.Timestamp(TRAIN_START)

tr_widx = np.where((win_ts>=train_start)&(win_ts<test_start))[0]
te_widx = np.where((win_ts>=test_start) &(win_ts<test_end))[0]
X_tr=windows[tr_widx]; X_te=windows[te_widx]
L_tr=l_arr[tr_widx+SEQ_LEN-1]; L_te=l_arr[te_widx+SEQ_LEN-1]
P_tr=prices_w[tr_widx]; P_te=prices_w[te_widx]
btc_s   = price_arr[(timestamps>=test_start)&(timestamps<test_end),0]
btc_ret = (btc_s[-1]/btc_s[0]-1)*100 if len(btc_s)>1 else 0.0

print(f"\n{'='*60}")
print(f"▶ {RUN_ROUND}  ~ {test_start.date()}  BTC: {btc_ret:+.1f}%")
print(f"  학습 {len(tr_widx):,}  테스트 {len(te_widx):,}")
print(f"{'='*60}")

p1_ckpt=f'{CKPT_DIR}/{RUN_ROUND}_phase1.pt'; p2_ckpt=f'{CKPT_DIR}/{RUN_ROUND}_phase2.pt'
oracle_cache=f'{CKPT_DIR}/{RUN_ROUND}_oracle.pkl'; vemb_cache=f'{CKPT_DIR}/{RUN_ROUND}_vembs.npy'
llm_cache=f'{CKPT_DIR}/{RUN_ROUND}_llmembs.npy'

# ── 모델 초기화 (l_dim=L_DIM=3) ──────────────────────────
model = VLAModel(
    n_coins=N_ASSETS, n_features=N_FEATURES, seq_len=SEQ_LEN,
    d_model=D_MODEL, n_heads=N_HEADS, n_layers_v=N_LAYERS,
    l_dim=L_DIM,        # 3 (pos/neg/score)
    n_heads_ca=1,       # l_dim=3이면 헤드 1개
    llm_name=LLM_NAME, lora_r=LORA_R, lora_alpha=LORA_ALPHA,
    n_assets=N_ASSETS, dropout=DROPOUT, device=device, use_lora=True,
).to(device)
model.print_params()

# Oracle 레이블
if os.path.exists(oracle_cache):
    d=joblib.load(oracle_cache); oracle_labels,focal_weights=d['labels'],d['focal']
    print('  Oracle 캐시 로드')
else:
    oracle=OracleLabelGenerator(commission=COMMISSION,n_assets=N_ASSETS,
        gamma=0.99,temperature=0.5,volatility_penalty=0.3,
        focal_weight=FOCAL_WEIGHT,focal_percentile=FOCAL_PERCENTILE)
    oracle_labels,focal_weights=oracle.generate(np.vstack([P_tr,P_tr[-1:]]))
    joblib.dump({'labels':oracle_labels,'focal':focal_weights},oracle_cache)

# Phase 1
if os.path.exists(p1_ckpt):
    model.load_state_dict(torch.load(p1_ckpt,map_location=device),strict=False)
    model.freeze_v_encoder(); print('  Phase 1 로드')
else:
    print(f'  Phase 1: Oracle SL ({ORACLE_EPOCHS} epochs)...')
    p1_params=(list(model.v_encoder.parameters())+list(model.cross_attn.parameters())+
               list(model.phase1_proj.parameters())+list(model.phase1_head.parameters())+
               list(model.v_norm.parameters())+list(model.l_norm.parameters()))
    opt1=torch.optim.AdamW(p1_params,lr=ORACLE_LR,weight_decay=1e-4)
    ds1=TensorDataset(torch.FloatTensor(X_tr),torch.FloatTensor(L_tr),
                       torch.FloatTensor(oracle_labels),torch.FloatTensor(focal_weights))
    dl1=DataLoader(ds1,batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True)
    for epoch in range(ORACLE_EPOCHS):
        model.train(); total=0
        for xb,lb,yb,fw in dl1:
            xb,lb,yb,fw=(t.to(device) for t in (xb,lb,yb,fw))
            w=model.forward_phase1(xb,lb)
            loss=(-(yb*torch.log(w.clamp(min=1e-7))).sum(-1)*fw).mean()
            if not torch.isfinite(loss): print(f'  ⚠️ NaN @ epoch {epoch+1}'); break
            opt1.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(p1_params,1.0); opt1.step(); total+=loss.item()
        if (epoch+1)%5==0: print(f'    Epoch {epoch+1:2d} | Loss: {total/len(dl1):.4f}')
    model.freeze_v_encoder(); torch.save(model.state_dict(),p1_ckpt)
    print('  ✅ Phase 1 저장')

# V 임베딩 사전계산
if os.path.exists(vemb_cache):
    v_embs_tr=np.load(vemb_cache)
else:
    model.eval()
    v_embs_tr=np.vstack([model.v_encoder(torch.FloatTensor(X_tr[i:i+64]).to(device))[2].cpu().numpy()
                          for i in range(0,len(X_tr),64)])
    np.save(vemb_cache,v_embs_tr)

# LLM 임베딩 사전계산 (뉴스 텍스트 대신 감성점수 사용)
if os.path.exists(llm_cache):
    llm_embs_tr=np.load(llm_cache); print('  LLM 임베딩 캐시 로드')
else:
    print('  LLM 임베딩 사전계산...')
    # 뉴스 텍스트 없이 v_emb + l_emb(감성점수)만으로 LLM 추론
    news_dummy = ['' ] * len(v_embs_tr)  # 텍스트 없음 (감성점수가 L)
    llm_embs_tr=model.precompute_llm_embs(v_embs_tr, L_tr, news_dummy)
    np.save(llm_cache,llm_embs_tr); print(f'  저장: {llm_embs_tr.shape}')

# Phase 2: GRPO
if os.path.exists(p2_ckpt):
    model.load_state_dict(torch.load(p2_ckpt,map_location=device),strict=False)
    print('  Phase 2 로드')
else:
    print(f'  Phase 2: GRPO ({GRPO_STEPS:,} steps)...')
    model.train()
    from training.grpo_trainer import GRPOTrainer
    grpo=GRPOTrainer(model,config,device=device)
    grpo.train(llm_embs_tr,np.vstack([P_tr,P_tr[-1:]]),rname=RUN_ROUND)
    torch.save(model.state_dict(),p2_ckpt); print('  ✅ Phase 2 저장')

# 백테스트
print('  백테스트...')
model.eval()
v_embs_te=np.vstack([model.v_encoder(torch.FloatTensor(X_te[i:i+64]).to(device))[2].cpu().numpy()
                      for i in range(0,len(X_te),64)])
news_dummy_te=['']*len(v_embs_te)
llm_embs_te=model.precompute_llm_embs(v_embs_te, L_te, news_dummy_te)

equity,prev_w=1.0,np.zeros(N_ASSETS+1); prev_w[-1]=1.0
with torch.no_grad():
    for i in range(len(llm_embs_te)-1):
        h_t=torch.FloatTensor(llm_embs_te[i]).unsqueeze(0).to(device)
        w=model.forward_from_llm_emb(h_t).squeeze(0).cpu().numpy()
        rets=(P_te[i+1]-P_te[i])/(P_te[i]+1e-9)
        equity*=(1+float(np.dot(w[:N_ASSETS],rets))-COMMISSION*np.abs(w-prev_w).sum())
        prev_w=w

total_ret=equity-1.0
result={'round':RUN_ROUND,'total_return':total_ret,'btc_ret_pct':float(btc_ret),
        'test_start':str(test_start.date()),'test_end':str(test_end.date())}
with open(RESULT_PATH,'w') as f: json.dump(result,f)
print(f'\n✅ {RUN_ROUND} | 전략: {total_ret*100:+.2f}%  BTC: {btc_ret:+.1f}%')
del model; gc.collect(); torch.cuda.empty_cache()

Device: cuda | NVIDIA A100-SXM4-40GB
V 피처: 106개 | 윈도우: 2257개


NameError: name 'L_DIM' is not defined

In [ ]:
# ─── CELL TRAIN-4: 병렬 실행 안내 ────────────────────────
# TRAIN-3의 RUN_ROUND 변수만 바꿔서 5개 세션 동시 실행:
#
#   세션 1: RUN_ROUND = 'R1'  → TRAIN-3 실행
#   세션 2: RUN_ROUND = 'R2'  → TRAIN-3 실행
#   세션 3: RUN_ROUND = 'R3'  → TRAIN-3 실행
#   세션 4: RUN_ROUND = 'R4'  → TRAIN-3 실행
#   세션 5: RUN_ROUND = 'R5'  → TRAIN-3 실행
#
# 결과는 각자 wf_results_rev4.json에 append 저장됨
print('병렬 실행: TRAIN-3의 RUN_ROUND 변수를 R1~R5로 바꿔서 5세션 동시 실행')

In [ ]:
# ─── 결과 확인 (학습 중단 없이 읽기만) ───────────────────
import json, os
path = f'{CKPT_DIR}/wf_results_rev4.json'
if os.path.exists(path):
    results = json.load(open(path))
    for r in results:
        print(f"{r['round']}: 전략 {r['total_return']*100:+.4f}%  BTC {r['btc_ret_pct']:+.1f}%")
else:
    print('결과 없음')

In [ ]:
# ─── CELL FIX-1: 수정된 파일들 재작성 (R3 끝난 후 실행) ──
# 변경사항:
# 1. Oracle: btc_ret 제거 (절대수익 기반) + Softmax 소프트 레이블 (T=0.5)
# 2. GRPO: std 최솟값 보장 (NaN 해결) + ActionHead만 학습
# 3. VLAModel: LLM 임베딩 사전계산 메서드 추가

import sys
for k in list(sys.modules.keys()):
    if any(x in k for x in ['vla_model','llm_reasoning','oracle_labels','grpo_trainer']):
        del sys.modules[k]

# ── oracle_labels.py ──────────────────────────────────────
code_oracle = '''import numpy as np
import torch
import torch.nn.functional as F

class OracleLabelGenerator:
    """
    Phase 1 GT 레이블 생성

    핵심 수정:
    1. btc_ret 제거 → 절대 수익률 기반 GT (최고 수익률 달성이 목표)
    2. Softmax 소프트 레이블 (Temperature T=0.5)
       - Hard 100%: 모델이 학습 불가 (너무 극단적)
       - Soft: Q값 비례 분포 → 모델이 경향성 학습 가능
       예) [SOL 50%, ETH 20%, BTC 15%, 현금 10%, ...]
    3. 수수료 패널티 유지 (자주 갈아타기 방지)
    4. 변동성 패널티 유지 (휩소 방지)
    """
    def __init__(self, commission=0.001, n_assets=5,
                 gamma=0.99, temperature=0.5,
                 volatility_penalty=0.3,
                 focal_weight=10.0, focal_percentile=95.0):
        self.comm        = commission
        self.n_assets    = n_assets
        self.gamma       = gamma
        self.temperature = temperature   # 소프트 레이블 온도
        self.vol_penalty = volatility_penalty
        self.focal_weight= focal_weight
        self.focal_pct   = focal_percentile
        self.n_actions   = n_assets + 1
        self._aw = np.array(
            [self._one_hot(i) for i in range(n_assets)] + [self._cash()],
            dtype=np.float32)

    def _cash(self):
        w = np.zeros(self.n_assets+1, dtype=np.float32); w[-1]=1.0; return w
    def _one_hot(self, idx):
        w = np.zeros(self.n_assets+1, dtype=np.float32); w[idx]=1.0; return w

    def _reward_matrix(self, prices, t, high=None, low=None):
        rets    = (prices[t+1]-prices[t])/(prices[t]+1e-9)
        # btc_ret 제거 — 절대 수익률 기반
        if high is not None and low is not None:
            mdd = (high[t]-low[t])/(prices[t]+1e-9)
        else:
            mdd = np.abs(rets)

        R = np.zeros((self.n_actions, self.n_actions), dtype=np.float32)
        for s in range(self.n_actions):
            for a in range(self.n_actions):
                w_new  = self._aw[a]; w_prev = self._aw[s]
                cost   = self.comm * np.abs(w_new-w_prev).sum()
                raw    = float(np.dot(w_new[:self.n_assets], rets))
                ep     = self.vol_penalty*float(mdd[a]) if a<self.n_assets else 0.0
                # 절대 수익 - 변동성 패널티 - 수수료 (BTC 비교 없음)
                R[s,a] = raw - ep - cost
        return R

    def compute_focal_weights(self, prices, high=None, low=None):
        T = len(prices)-1
        if high is None: return np.ones(T, dtype=np.float32)
        mdd = np.mean((high[:T]-low[:T])/(prices[:T]+1e-9), axis=1)
        thr = np.percentile(mdd, self.focal_pct)
        return np.where(mdd>=thr, self.focal_weight, 1.0).astype(np.float32)

    def generate(self, prices, high=None, low=None, init_action=None, verbose=True):
        T  = len(prices)-1
        A  = self.n_actions
        s0 = init_action if init_action is not None else self.n_assets

        if verbose: print(f"  Oracle DP: T={T}")
        R_all = np.stack([self._reward_matrix(prices,t,high,low) for t in range(T)])

        # Backward DP
        V = np.zeros(A, dtype=np.float32)
        Q_all = np.zeros((T, A, A), dtype=np.float32)
        for t in range(T-1,-1,-1):
            Q = R_all[t] + self.gamma*V[np.newaxis,:]
            Q_all[t] = Q; V = Q.max(axis=1)

        # Forward 패스 + Softmax 소프트 레이블
        labels = np.zeros((T, self.n_assets+1), dtype=np.float32)
        cur    = s0
        for t in range(T):
            q_t = Q_all[t, cur]   # 현재 상태에서 각 액션의 Q값

            # Temperature Softmax: Q값 비례 소프트 레이블
            q_tensor = torch.tensor(q_t / self.temperature, dtype=torch.float32)
            probs    = F.softmax(q_tensor, dim=0).numpy()

            # 소프트 레이블: Q값에 비례한 포트폴리오 비중 가중합
            label = (probs[:, np.newaxis] * self._aw).sum(axis=0)
            label = label / (label.sum() + 1e-8)
            labels[t] = label

            # 다음 상태: 가장 높은 Q값 액션으로 이동
            cur = q_t.argmax()

        fw = self.compute_focal_weights(prices, high, low)

        if verbose:
            cash_r = (labels[:,-1] > 0.3).mean()*100
            n_f    = (fw > 1.0).sum()
            print(f"  완료: 현금비중>30% {cash_r:.1f}% | Focal {n_f}/{T} "
                  f"| Temperature={self.temperature}")

        return labels, fw
'''
with open(f'{REV4_SRC}/training/oracle_labels.py', 'w') as f:
    f.write(code_oracle)
print('✅ oracle_labels.py')
print('   - btc_ret 제거 (절대수익 기반)')
print('   - Softmax 소프트 레이블 T=0.5')

# ── vla_model.py: LLM 사전계산 추가 ─────────────────────
code_vla = '''import torch, torch.nn as nn
import numpy as np
from models.itransformer import iTransformer
from models.cross_attention import CrossModalAttention
from models.llm_reasoning import LLMReasoningModule
from models.action_head import ActionHead

class VLAModel(nn.Module):
    def __init__(self, n_coins=5, n_features=106, seq_len=60,
                 d_model=256, n_heads=8, n_layers_v=4,
                 l_dim=768, n_heads_ca=8,
                 llm_name="Qwen/Qwen2.5-1.5B",
                 lora_r=16, lora_alpha=32,
                 n_assets=5, dropout=0.1,
                 device="cpu", use_lora=True):
        super().__init__()
        self.device   = device
        self.n_assets = n_assets
        self.v_encoder  = iTransformer(n_coins=n_coins, n_features=n_features,
            seq_len=seq_len, d_model=d_model, n_heads=n_heads,
            n_layers=n_layers_v, dropout=dropout)
        self.cross_attn = CrossModalAttention(v_dim=d_model, l_dim=l_dim,
            d_model=d_model, n_heads=n_heads_ca, dropout=dropout)
        self.phase1_proj = nn.Sequential(nn.Linear(d_model, d_model*2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model*2, d_model))
        self.phase1_head = ActionHead(d_model, n_assets, dropout)
        self.llm = LLMReasoningModule(v_dim=d_model, llm_name=llm_name,
            device=device, use_lora=use_lora, lora_r=lora_r, lora_alpha=lora_alpha)
        self.action_head = ActionHead(self.llm.hidden_size, n_assets, dropout)
        self.v_norm = nn.LayerNorm(d_model)
        self.l_norm = nn.LayerNorm(l_dim)

    def forward_phase1(self, x, l_emb):
        _,_,v = self.v_encoder(x.to(self.device))
        v = self.v_norm(v)
        l = self.l_norm(l_emb.to(self.device))
        fused,_,_ = self.cross_attn(v, l)
        return self.phase1_head(self.phase1_proj(fused))

    def precompute_llm_embs(self, v_embs, l_embs, news_texts, batch_size=8):
        """LLM 임베딩 사전계산 — GRPO 속도 50배↑"""
        for p in self.llm.parameters(): p.requires_grad_(False)
        self.eval()
        all_h = []
        with torch.no_grad():
            for i in range(0, len(v_embs), batch_size):
                v_b = torch.FloatTensor(v_embs[i:i+batch_size]).to(self.device)
                l_b = torch.FloatTensor(l_embs[i:i+batch_size]).to(self.device)
                v_n = self.v_norm(v_b); l_n = self.l_norm(l_b)
                fused,_,_ = self.cross_attn(v_n, l_n)
                h = self.llm(fused, news_texts[i:i+batch_size])
                all_h.append(h.cpu().numpy())
        return np.vstack(all_h)

    def forward_from_llm_emb(self, llm_emb, nav=None):
        return self.action_head(llm_emb.to(self.device), nav)

    def forward_from_v_emb(self, v_emb, l_emb, news_texts, nav=None):
        v = self.v_norm(v_emb.to(self.device))
        l = self.l_norm(l_emb.to(self.device))
        fused,attn,_ = self.cross_attn(v, l)
        h = self.llm(fused, news_texts)
        return self.action_head(h, nav), h, attn

    def freeze_v_encoder(self):
        for p in self.v_encoder.parameters(): p.requires_grad_(False)
        print("✅ iTransformer 동결")

    def trainable_parameters(self):
        return [p for p in self.parameters() if p.requires_grad]

    def print_params(self):
        total = sum(p.numel() for p in self.parameters())
        tr    = sum(p.numel() for p in self.trainable_parameters())
        print(f"  iTransformer:  {sum(p.numel() for p in self.v_encoder.parameters()):,}")
        print(f"  CrossAttn:     {sum(p.numel() for p in self.cross_attn.parameters()):,}")
        print(f"  LLM LoRA:      {sum(p.numel() for p in self.llm.parameters() if p.requires_grad):,}")
        print(f"  전체: {total:,}  학습: {tr:,} ({100*tr/total:.1f}%)")
'''
with open(f'{REV4_SRC}/models/vla_model.py', 'w') as f:
    f.write(code_vla)
open(f'{REV4_SRC}/models/__init__.py','w').close()
print('✅ vla_model.py — precompute_llm_embs 추가')

# ── grpo_trainer.py: ActionHead만, std 최솟값 ────────────
code_grpo = '''import torch, torch.nn as nn, copy, numpy as np
from torch.distributions import Dirichlet
from tqdm import tqdm

class GRPOTrainer:
    """GRPO — LLM 임베딩 사전계산 활용, ActionHead만 학습"""
    def __init__(self, model, config, device="cuda"):
        self.model  = model
        self.cfg    = config
        self.device = device
        self.opt    = torch.optim.AdamW(
            model.action_head.parameters(), lr=config.GRPO_LR, weight_decay=1e-4)
        self.ref_weights  = None
        self._ret_history = []

    def _init_ref(self):
        self.ref_weights = copy.deepcopy(self.model.action_head.state_dict())

    def _ref_forward(self, h_t):
        orig = copy.deepcopy(self.model.action_head.state_dict())
        self.model.action_head.load_state_dict(self.ref_weights)
        with torch.no_grad():
            w = self.model.forward_from_llm_emb(h_t)
        self.model.action_head.load_state_dict(orig)
        return w

    def _rewards(self, samples, prices, t, prev_w):
        rets    = (prices[t+1]-prices[t])/(prices[t]+1e-9)
        btc_ret = float(rets[0])
        lam     = getattr(self.cfg,'SORTINO_LAMBDA',0.1)
        dv = 0.0
        if len(self._ret_history)>5:
            r=np.array(self._ret_history[-50:]); d=r[r<0]
            dv = float(np.std(d)) if len(d)>1 else 0.0
        rews = []
        for w in samples.cpu().numpy():
            p = float(np.dot(w[:len(rets)],rets))
            c = self.cfg.COMMISSION*np.abs(w-prev_w).sum()
            rews.append(p-btc_ret-c-lam*dv)
            self._ret_history.append(p-btc_ret)
        return torch.tensor(rews, dtype=torch.float32, device=self.device)

    def _loss(self, lp, ref_lp, rewards):
        std = rewards.std().clamp(min=1e-3)  # NaN 방지
        adv = (rewards-rewards.mean())/(std+1e-8)
        ratio   = torch.exp(lp-ref_lp)
        clipped = ratio.clamp(1-self.cfg.GRPO_CLIP_EPS, 1+self.cfg.GRPO_CLIP_EPS)
        return (-torch.min(ratio*adv, clipped*adv).mean()
                + self.cfg.GRPO_KL_BETA*(lp-ref_lp).mean())

    def train(self, llm_embs, prices, n_steps=None, rname=""):
        """llm_embs: (T, H) 사전계산 — ActionHead만 학습 → 매우 빠름"""
        if self.ref_weights is None: self._init_ref()
        n_steps = n_steps or self.cfg.GRPO_STEPS
        T, G    = len(llm_embs)-1, self.cfg.GRPO_GROUP_SIZE
        prev_w  = np.zeros(prices.shape[1]+1); prev_w[-1]=1.0
        total   = 0.0; self._ret_history=[]
        self.model.action_head.train()
        pbar = tqdm(range(n_steps), desc=f"GRPO {rname}")
        for step in pbar:
            t   = np.random.randint(0, T)
            h_t = torch.FloatTensor(llm_embs[t]).unsqueeze(0).to(self.device)
            w   = self.model.forward_from_llm_emb(h_t).squeeze(0)
            conc= (w*self.cfg.GRPO_DIRICHLET_CONC).clamp(min=0.1)
            dist= Dirichlet(conc); samp=dist.sample((G,)); lp=dist.log_prob(samp)
            with torch.no_grad():
                rw  = self._ref_forward(h_t).squeeze(0)
                rc  = (rw*self.cfg.GRPO_DIRICHLET_CONC).clamp(min=0.1)
                rlp = Dirichlet(rc).log_prob(samp)
            rew  = self._rewards(samp,prices,t,prev_w)
            loss = self._loss(lp,rlp,rew)
            self.opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(self.model.action_head.parameters(),1.0)
            self.opt.step()
            prev_w=w.detach().cpu().numpy(); total+=loss.item()
            if (step+1)%1000==0:
                pbar.set_postfix({"loss":f"{total/(step+1):.4f}",
                                  "rew":f"{rew.mean().item():.4f}"})
        return total/n_steps
'''
with open(f'{REV4_SRC}/training/grpo_trainer.py', 'w') as f:
    f.write(code_grpo)
print('✅ grpo_trainer.py — ActionHead만, std 최솟값, NaN 해결')
print()
print('변경 요약:')
print('  Oracle: btc_ret 제거 + Softmax T=0.5 소프트 레이블')
print('  GRPO:   LLM 사전계산 활용 → 2시간 → ~5분')

✅ oracle_labels.py
   - btc_ret 제거 (절대수익 기반)
   - Softmax 소프트 레이블 T=0.5
✅ vla_model.py — precompute_llm_embs 추가
✅ grpo_trainer.py — ActionHead만, std 최솟값, NaN 해결

변경 요약:
  Oracle: btc_ret 제거 + Softmax T=0.5 소프트 레이블
  GRPO:   LLM 사전계산 활용 → 2시간 → ~5분


In [ ]:
# ─── CELL FIX-2: R3 완료 후 실행 ─────────────────────────
# 1. wf_results_rev4.json에서 R2 제거
# 2. TRAIN-3 수정사항 적용 후 R2/R4/R5 실행 준비

import json, os

RESULT_PATH = f'{CKPT_DIR}/wf_results_rev4.json'
results = json.load(open(RESULT_PATH))

# R2 제거 (NaN)
before = len(results)
results = [r for r in results if r['round'] != 'R2']
with open(RESULT_PATH, 'w') as f: json.dump(results, f)

print(f'wf_results_rev4.json: {before}개 → {len(results)}개 (R2 제거)')
print(f'현재 완료 라운드: {[r["round"] for r in results]}')

# R2 phase2 체크포인트 제거 (phase1은 유지)
p2 = f'{CKPT_DIR}/R2_phase2.pt'
if os.path.exists(p2):
    os.remove(p2); print('R2_phase2.pt 삭제 (phase1은 유지)')

print('\n다음: TRAIN-3 수정 후 병렬 실행')
print('  세션1: RUN_ROUND = "R2"')
print('  세션2: RUN_ROUND = "R4"')
print('  세션3: RUN_ROUND = "R5"')

In [ ]:
# ─── CELL FIX-3: 기존 체크포인트 전체 초기화 ─────────────
import os, glob

# 결과 파일 초기화
result_path = f'{CKPT_DIR}/wf_results_rev4.json'
if os.path.exists(result_path):
    os.remove(result_path)
    print('✅ wf_results_rev4.json 삭제')

# 모든 체크포인트 삭제
for f in glob.glob(f'{CKPT_DIR}/*.pt') + \
         glob.glob(f'{CKPT_DIR}/*.npy') + \
         glob.glob(f'{CKPT_DIR}/*.pkl'):
    os.remove(f)
    print(f'  삭제: {os.path.basename(f)}')

print('\n✅ 초기화 완료 — 처음부터 다시 학습')

✅ wf_results_rev4.json 삭제
  삭제: R1_phase1.pt
  삭제: R1_phase2.pt
  삭제: R2_phase1.pt
  삭제: R2_phase2.pt
  삭제: R3_phase1.pt
  삭제: R1_vembs.npy
  삭제: R2_vembs.npy
  삭제: R3_vembs.npy
  삭제: finbert_embeddings.pkl
  삭제: R1_oracle.pkl
  삭제: R2_oracle.pkl
  삭제: R3_oracle.pkl

✅ 초기화 완료 — 처음부터 다시 학습


In [ ]:
# ─── CELL RESULT: 결과 병합 + 분석 ───────────────────────
import json, os, numpy as np
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec

CKPT_DIR = '/content/drive/MyDrive/X-MultiVLA_rev4/checkpoints'
OUT_DIR  = '/content/drive/MyDrive/X-MultiVLA_rev4/outputs'
os.makedirs(OUT_DIR, exist_ok=True)

# 라운드별 파일 병합
ALL_ROUNDS = ['R1','R2','R3','R4','R5']
results = []
for rname in ALL_ROUNDS:
    path = f'{CKPT_DIR}/wf_results_{rname}.json'
    if os.path.exists(path):
        results.append(json.load(open(path)))
        print(f'  ✅ {rname} 로드')
    else:
        print(f'  ⏳ {rname} 아직 없음')

if not results:
    print('결과 없음'); raise SystemExit

print(f"\n{'='*65}")
print(f"  X-MultiVLA Rev4 — Walk-Forward 결과")
print(f"{'='*65}")
print(f"  {'라운드':<5} {'기간':<23} {'전략':>8} {'BTC':>9} {'알파':>8}")
print(f"  {'-'*63}")

strat, btc_rets, alphas = [], [], []
for r in results:
    s=r['total_return']*100; b=r['btc_ret_pct']; a=s-b
    strat.append(s); btc_rets.append(b); alphas.append(a)
    flag='✅' if a>0 else '❌'
    print(f"  {flag} {r['round']:<4} {r['test_start']}~{r['test_end']}  "
          f"{s:>+7.2f}%  {b:>+7.1f}%  {a:>+7.2f}%")

if len(results)>1:
    print(f"  {'-'*63}")
    print(f"  {'평균':<28} {np.mean(strat):>+7.2f}%  {np.mean(btc_rets):>+7.1f}%  {np.mean(alphas):>+7.2f}%")
print(f"{'='*65}")

beat = sum(1 for a in alphas if a>0)
print(f"  BTC 초과 달성: {beat}/{len(results)}라운드")

# 시각화
if len(results)>=2:
    fig=plt.figure(figsize=(13,8)); gs=gridspec.GridSpec(2,2,hspace=0.45,wspace=0.3)
    labels=[r['round'] for r in results]; x=np.arange(len(results)); w=0.35
    ax1=fig.add_subplot(gs[0,:])
    b1=ax1.bar(x-w/2,strat,w,label='전략',color='steelblue')
    b2=ax1.bar(x+w/2,btc_rets,w,label='BTC B&H',color='orange',alpha=0.8)
    ax1.axhline(0,color='black',lw=0.8,ls='--')
    if len(strat)>1: ax1.axhline(np.mean(strat),color='steelblue',lw=1.5,ls=':',
                                   label=f'전략평균 {np.mean(strat):+.1f}%')
    for b in [*b1,*b2]:
        h=b.get_height()
        ax1.text(b.get_x()+b.get_width()/2,h+(0.3 if h>=0 else -1.5),
                 f'{h:+.1f}%',ha='center',va='bottom',fontsize=8)
    ax1.set_xticks(x); ax1.set_xticklabels(labels)
    ax1.set_title('라운드별 수익률'); ax1.legend(); ax1.grid(axis='y',alpha=0.3)

    ax2=fig.add_subplot(gs[1,0])
    ax2.bar(labels,alphas,color=['green' if a>0 else 'red' for a in alphas],alpha=0.8)
    ax2.axhline(0,color='black',lw=0.8)
    ax2.set_title('BTC 대비 알파'); ax2.grid(axis='y',alpha=0.3)

    ax3=fig.add_subplot(gs[1,1])
    rev3={'R1':-11.31,'R2':12.07,'R3':-34.82,'R4':-29.74,'R5':-0.57}
    rv=[rev3.get(r['round'],0) for r in results]
    ax3.bar(x-w/2,rv,w,label='Rev3',color='tomato',alpha=0.7)
    ax3.bar(x+w/2,strat,w,label='Rev4',color='steelblue',alpha=0.8)
    ax3.axhline(0,color='black',lw=0.8)
    ax3.set_xticks(x); ax3.set_xticklabels(labels)
    ax3.set_title('Rev3 vs Rev4'); ax3.legend(); ax3.grid(axis='y',alpha=0.3)

    plt.suptitle('X-MultiVLA Rev4',fontsize=13,fontweight='bold')
    plt.savefig(f'{OUT_DIR}/rev4_results.png',dpi=150,bbox_inches='tight')
    plt.show(); print(f'💾 {OUT_DIR}/rev4_results.png')

In [ ]:
# ─── CELL 20: 데이터 준비 ─────────────────────────────────
# 1) 뉴스 데이터 Rev3 → Rev4 복사
# 2) 새 바이낸스 파일은 직접 업로드 후 경로 지정
import os, shutil

REV3_DATA = f'{REV3_ROOT}/data'
REV4_DATA = f'{REV4_ROOT}/data'

# ── 뉴스 데이터 복사 ─────────────────────────────────────
for fname in ['news_all_8h.csv', 'news_sentiment_8h.csv']:
    src = f'{REV3_DATA}/{fname}'
    dst = f'{REV4_DATA}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f'✅ 복사 완료: {fname}')
    elif os.path.exists(dst):
        print(f'✅ 이미 존재: {fname}')
    else:
        print(f'❌ 없음: {fname}')

# ── 새 바이낸스 파일 확인 ────────────────────────────────
print('\n현재 Rev4/data 파일 목록:')
for f in sorted(os.listdir(REV4_DATA)):
    size = os.path.getsize(f'{REV4_DATA}/{f}') / 1024 / 1024
    print(f'  {f}  ({size:.1f} MB)')

print('\n새 바이낸스 parquet 파일을 아래 경로에 업로드해줘:')
print(f'  {REV4_DATA}/')
print('\n업로드 후 아래 변수에 파일명 입력:')
NEW_FUTURES_FILE = None  # 예: 'binance_futures_rev4.parquet'
if NEW_FUTURES_FILE:
    FUTURES_PATH = f'{REV4_DATA}/{NEW_FUTURES_FILE}'
    print(f'✅ 사용할 파일: {FUTURES_PATH}')
else:
    print('⏳ 파일 업로드 후 NEW_FUTURES_FILE 변수에 파일명 입력')

In [ ]:
# ─── CELL 21: FinBERT 768d 임베딩 사전계산 + 저장 ─────────
# 뉴스 데이터를 한 번만 FinBERT로 처리해서 저장
# 이후 학습 시 매번 FinBERT 돌릴 필요 없음
import pandas as pd
import numpy as np
import torch
import joblib
import os
from transformers import AutoTokenizer, AutoModel

EMBED_SAVE_PATH = f'{REV4_DATA}/news_finbert_768d.pkl'

if os.path.exists(EMBED_SAVE_PATH):
    print(f'✅ 이미 계산된 임베딩 존재: {EMBED_SAVE_PATH}')
    print('   재계산 필요하면 파일 삭제 후 재실행')
else:
    print('FinBERT 768d 임베딩 계산 중...')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
    model = AutoModel.from_pretrained('ProsusAI/finbert').to(device)
    model.eval()

    news = pd.read_csv(f'{REV4_DATA}/news_sentiment_8h.csv',
                       parse_dates=['published_at'], encoding='utf-8')
    print(f'  뉴스 {len(news):,}건 로드')

    BATCH = 64
    all_embs = []

    for i in range(0, len(news), BATCH):
        batch_texts = news['title'].iloc[i:i+BATCH].fillna('').tolist()
        enc = tokenizer(batch_texts, padding=True, truncation=True,
                        max_length=128, return_tensors='pt').to(device)
        with torch.no_grad():
            out = model(**enc)
        embs = out.last_hidden_state[:, 0, :].cpu().numpy()  # CLS (B, 768)
        all_embs.append(embs)

        if (i // BATCH) % 50 == 0:
            print(f'  진행: {i:,}/{len(news):,}')

    all_embs = np.vstack(all_embs)  # (N, 768)

    # 타임스탬프 + 코인 정보와 함께 저장
    result = {
        'embeddings': all_embs,              # (N, 768)
        'published_at': news['published_at'].values,
        'coin': news['coin'].values,
        'datetime_8h': pd.to_datetime(news['published_at']).dt.floor('8h').values,
    }
    joblib.dump(result, EMBED_SAVE_PATH)
    print(f'\n✅ 저장 완료: {EMBED_SAVE_PATH}')
    print(f'   shape: {all_embs.shape}')
    print(f'   크기: {os.path.getsize(EMBED_SAVE_PATH)/1024/1024:.1f} MB')